# EDA Round 3 – Kiểm tra dữ liệu bẩn và lỗi

Notebook này kiểm tra 8 file trong thư mục `Round3_BI10_dataset`. Mục tiêu là tìm chỗ dữ liệu bị trùng, bị thiếu, ghi không thống nhất, hoặc dễ hiểu nhầm, để cả nhóm thống nhất cách xử lý trước khi phân tích.

Mỗi lần kiểm tra đều đi theo thứ tự: nói trước sẽ kiểm tra điều gì và câu lệnh làm gì, sau đó chạy câu lệnh, cuối cùng đọc kết quả.

**Cách chạy:**
- Đặt notebook này cùng thư mục với `Round3_BI10_dataset`.
- Cài thư viện: `pip install duckdb pandas openpyxl`.
- Chạy từ trên xuống. Lần đầu, notebook đọc toàn bộ file CSV vào một file `r3_eda.duckdb` (khoảng 750 MB, mất chừng 1 phút). Những lần sau đọc lại file này nên nhanh hơn. Có thể xóa file đó khi không cần nữa.

Notebook dùng **DuckDB**, một công cụ chạy câu lệnh SQL ngay trên máy, không cần cài máy chủ. Lý do: file `leads.csv` nặng hơn 1 GB và `contract.csv` gần 600 MB, đọc bằng pandas thuần sẽ chậm và tốn RAM.

Ô dưới đây chuẩn bị dữ liệu:
- đọc 4 file CSV và 3 file Excel thành 7 bảng tên `contract`, `leads`, `cp` (contract_product), `e2s` (emp_2_salesroom), `salesroom`, `seller`, `reason` (cl_lead_fin_reason);
- tạo hàm `q()` để chạy một câu SQL và trả kết quả ra dạng bảng;
- tạo biến `SIGNED` chứa 5 trạng thái được tính là đã ký: Active, Finished, Paid off, Written off, Signed. Đây là quy ước trong StructureTable của Round 2.

In [ ]:
import os, tempfile, duckdb, pandas as pd
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)
pd.set_option('display.max_rows', 80); pd.set_option('display.max_colwidth', 120); pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

DATA = 'Round3_BI10_dataset'
DB = 'r3_eda.duckdb'
TMP = os.path.join(tempfile.gettempdir(), 'duck_r3'); os.makedirs(TMP, exist_ok=True)

def build():
    con = duckdb.connect(DB)
    con.execute(f"SET threads=2; SET memory_limit='2GB'; SET temp_directory='{TMP}'; SET preserve_insertion_order=false")
    con.execute(f"CREATE OR REPLACE TABLE contract AS SELECT * FROM read_csv('{DATA}/contract.csv')")
    con.execute(f"CREATE OR REPLACE TABLE leads AS SELECT * FROM read_csv('{DATA}/leads.csv', types={{'code_pos':'VARCHAR','code_sa':'VARCHAR','text_contract_number':'VARCHAR','text_tenor':'VARCHAR','text_accepted_by':'VARCHAR','code_lead_status':'VARCHAR'}})")
    con.execute(f"CREATE OR REPLACE TABLE cp AS SELECT * FROM read_csv('{DATA}/contract_product.csv')")
    con.execute(f"CREATE OR REPLACE TABLE e2s AS SELECT * FROM read_csv('{DATA}/emp_2_salesroom.csv')")
    for f, t in [('salesroom.xlsx', 'salesroom'), ('seller.xlsx', 'seller'), ('cl_lead_fin_reason.xlsx', 'reason')]:
        d = pd.read_excel(f'{DATA}/{f}', dtype=str); con.register('d', d)
        con.execute(f"CREATE OR REPLACE TABLE {t} AS SELECT * FROM d"); con.unregister('d')
    con.close()

if not os.path.exists(DB):
    build()
con = duckdb.connect(DB, read_only=True)
con.execute(f"SET threads=2; SET memory_limit='2GB'; SET temp_directory='{TMP}'; SET preserve_insertion_order=false")
def q(sql): return con.execute(sql).df()
SIGNED = "('Active','Finished','Paid off','Written off','Signed')"

## Phần 1. Mỗi bảng có bao nhiêu dòng và phủ khoảng thời gian nào

Nếu các bảng không cùng khoảng thời gian, những phép so sánh giữa bảng (ví dụ nối lead với contract) chỉ làm được ở đoạn thời gian chung. Vì vậy việc đầu tiên là xem mỗi bảng bắt đầu và kết thúc khi nào.

**Kiểm tra 1.1. Khoảng thời gian của từng bảng.** Câu lệnh dưới đây đếm số dòng của mỗi bảng, rồi lấy ngày sớm nhất (cột `tu`) và muộn nhất (cột `den`) theo cột ngày chính của bảng đó. Cột `cot` ghi tên cột ngày được dùng.

In [ ]:
q('''
select 'contract' bang, count(*) so_dong, min(date_apply)::date tu, max(date_apply)::date den, 'date_apply' cot from contract
union all select 'leads', count(*), min(dtime_created)::date, max(dtime_created)::date, 'dtime_created' from leads
union all select 'emp_2_salesroom', count(*), min(dtime_valid_from)::date, max(dtime_valid_from)::date, 'dtime_valid_from' from e2s
union all select 'salesroom', count(*), min(strptime(SALESROOM_START,'%d-%m-%Y %H:%M:%S'))::date, max(strptime(SALESROOM_START,'%d-%m-%Y %H:%M:%S'))::date, 'SALESROOM_START' from salesroom
union all select 'seller', count(*), min(strptime(SELLER_ACTIVE_DATE,'%d-%m-%Y %H:%M:%S'))::date, max(strptime(SELLER_ACTIVE_DATE,'%d-%m-%Y %H:%M:%S'))::date, 'SELLER_ACTIVE_DATE' from seller
union all select 'contract_product', count(*), null, null, '(không có cột ngày)' from cp
''')

,bang,so_dong,tu,den,cot
0,contract,3775371,2025-01-01,2026-09-30,date_apply
1,leads,6130637,2025-12-31,2026-10-17,dtime_created
2,emp_2_salesroom,275856,2025-01-02,2026-10-17,dtime_valid_from
3,salesroom,14231,2025-01-02,2026-10-18,SALESROOM_START
4,seller,11639,2008-09-01,2026-10-16,SELLER_ACTIVE_DATE
5,contract_product,784902,NaT,NaT,(không có cột ngày)


Contract có hồ sơ nộp từ 01/01/2025 đến 30/09/2026, nhưng lead chỉ bắt đầu từ 31/12/2025. Như vậy phần **cross-sell** (bán chéo, tức lead của shop dẫn tới contract ở kênh khác) chỉ tính được cho 01–09/2026. Năm 2025 không có lead để nối.

Salesroom có ngày bắt đầu sớm nhất là 02/01/2025, trong khi seller có đối tác hoạt động từ năm 2008. Vì vậy `SALESROOM_START` chưa chắc là ngày shop thật sự mở cửa.

**Kiểm tra 1.2. Lead có đủ ngày trong từng tháng không.** Nếu một tháng chỉ có vài ngày dữ liệu, số lead của tháng đó sẽ thấp bất thường khi vẽ biểu đồ theo tháng. Câu lệnh dưới đây, với mỗi tháng, đếm:
- số lead;
- số ngày khác nhau có lead (`so_ngay_co_du_lieu`);
- ngày đầu tiên và ngày cuối cùng có lead trong tháng.

In [ ]:
q('''select strftime(date_trunc('month',dtime_created),'%Y-%m') thang, count(*) so_dong_lead,
  count(distinct dtime_created::date) so_ngay_co_du_lieu, min(dtime_created)::date ngay_dau, max(dtime_created)::date ngay_cuoi
from leads group by 1 order by 1''')

,thang,so_dong_lead,so_ngay_co_du_lieu,ngay_dau,ngay_cuoi
0,2025-12,18352,1,2025-12-31,2025-12-31
1,2026-01,668277,31,2026-01-01,2026-01-31
2,2026-02,596567,28,2026-02-01,2026-02-28
3,2026-03,819796,31,2026-03-01,2026-03-31
4,2026-04,526539,30,2026-04-01,2026-04-30
5,2026-05,522345,31,2026-05-01,2026-05-31
6,2026-06,492565,30,2026-06-01,2026-06-30
7,2026-07,555148,31,2026-07-01,2026-07-31
8,2026-08,700065,31,2026-08-01,2026-08-31
9,2026-09,848681,30,2026-09-01,2026-09-30


Tháng 12/2025 chỉ có 1 ngày dữ liệu (31/12), và tháng 10/2026 chỉ có 17 ngày (01/10 đến 17/10). Các tháng còn lại đủ ngày.

Khi so sánh lead theo tháng, hai tháng này sẽ thấp hơn thực tế. Cần bỏ hai tháng này, hoặc ghi chú rõ trên biểu đồ.

## Phần 2. Bảng seller

**Kiểm tra 2.1. Mã seller có trùng không, có cột nào bị trống không.** Câu lệnh dưới đây so số dòng với số mã `SKP_SELLER` khác nhau. Nếu hai số bằng nhau thì mã không trùng. Các cột `trong_...` đếm số ô trống của từng cột.

In [ ]:
q('''select count(*) so_dong, count(distinct SKP_SELLER) so_ma_khac_nhau,
  count(*)-count(SELLER_MODEL) trong_model, count(*)-count(SELLER_STATUS) trong_status, count(*)-count(SELLER_TYPE) trong_type,
  count(*)-count(SELLER_PRODUCT) trong_product, count(*)-count(SELLER_PROVINCE) trong_province, count(*)-count(SELLER_AREA) trong_area
from seller''')

,so_dong,so_ma_khac_nhau,trong_model,trong_status,trong_type,trong_product,trong_province,trong_area
0,11639,11639,0,0,0,0,0,0


Số dòng bằng số mã khác nhau (11.639), và không có ô trống nào. Bảng seller sạch về mặt này.

**Kiểm tra 2.2. Tên tỉnh có còn ghi theo địa giới cũ không.** Từ 01/07/2025, theo Nghị quyết 202/2025, cả nước sáp nhập từ 63 xuống 34 tỉnh, thành. Biến `TINH_CU` dưới đây liệt kê 20 tỉnh đã bị sáp nhập vào tỉnh khác, tức những tên không còn tồn tại theo địa giới mới. Câu lệnh đếm xem còn bao nhiêu seller ghi các tên này.

In [ ]:
TINH_CU = "('Bình Định','Bạc Liêu','Bắc Giang','Bắc Kạn','Bến Tre','Hà Giang','Hà Nam','Hải Dương','Kiên Giang','Kon Tum','Nam Định','Ninh Thuận','Phú Yên','Quảng Bình','Quảng Nam','Tiền Giang','Trà Vinh','Vĩnh Phúc','Yên Bái','Đắk Nông')"
q(f'''select SELLER_PROVINCE, count(*) so_seller from seller where SELLER_PROVINCE in {TINH_CU} group by 1 order by 2 desc''')

,SELLER_PROVINCE,so_seller
0,Kiên Giang,22
1,Bắc Giang,9
2,Tiền Giang,8
3,Quảng Bình,6
4,Quảng Nam,5
5,Ninh Thuận,5
6,Trà Vinh,4
7,Vĩnh Phúc,4
8,Bình Định,4
9,Nam Định,4


Cả 20 tên tỉnh cũ vẫn còn xuất hiện, ví dụ Kiên Giang (22 seller), Bắc Giang (9), Tiền Giang (8). Mỗi tên chỉ có vài seller, nhưng nếu ghép với dữ liệu dân số theo 34 tỉnh mới thì phải đổi các tên này về tỉnh mới trước.

**Kiểm tra 2.3. Model và ngành hàng có liên quan với nhau không.** Câu lệnh dưới đây đếm số seller theo từng cặp model (Aldi hoặc Non-Aldi) và ngành hàng (CD, NS, TW).

In [ ]:
q('''select SELLER_MODEL, SELLER_PRODUCT, count(*) so_seller from seller group by all order by all''')

,SELLER_MODEL,SELLER_PRODUCT,so_seller
0,Aldi,NS,3094
1,Aldi,TW,1
2,Non-Aldi,CD,6509
3,Non-Aldi,NS,1
4,Non-Aldi,TW,2034


Có 3.094 trên 3.095 seller Aldi bán NS, và gần như mọi seller NS đều là Aldi. Hai cột này gần như trùng nhau.

Hệ quả: khi phân tích, phép so sánh "Aldi với Non-Aldi" gần như chính là phép so sánh "NS với CD/TW". Không thể tách riêng ảnh hưởng của mô hình làm hồ sơ và ảnh hưởng của ngành hàng.

## Phần 3. Bảng salesroom

**Kiểm tra 3.1. Mã shop có trùng không, cột nào bị trống.** Câu lệnh dưới đây so số dòng với số mã `SKP_SALESROOM` khác nhau, và đếm ô trống ở một số cột.

In [ ]:
q('''select count(*) so_dong, count(distinct SKP_SALESROOM) so_ma_khac_nhau, count(*)-count(SALESROOM_END) trong_end,
  count(*)-count(SALESROOM_TOWN) trong_town, count(*)-count(SKP_SELLER) trong_seller from salesroom''')

,so_dong,so_ma_khac_nhau,trong_end,trong_town,trong_seller
0,14231,14231,12030,0,0


Mã shop không trùng. Có 12.030 shop không có ngày đóng.

**Kiểm tra 3.2. Ngày đóng có khớp với trạng thái không.** Theo mô tả của Ban tổ chức, ngày đóng để trống nếu shop còn hoạt động. Câu lệnh dưới đây đếm số shop theo từng trạng thái, tách theo ngày đóng có trống (`end_trong = True`) hay không.

In [ ]:
q('''select SALESROOM_STATUS, SALESROOM_END is null as end_trong, count(*) so_shop from salesroom group by all order by all''')

,SALESROOM_STATUS,end_trong,so_shop
0,Active,True,11847
1,Blocked,True,179
2,Closed,False,2201
3,On activation,True,1
4,Refused,True,3


Ngày đóng khớp với trạng thái:
- Shop Closed luôn có ngày đóng.
- Các trạng thái còn lại (Active, Blocked, Refused, On activation) đều để trống ngày đóng.

**Kiểm tra 3.3. Shop có nối đúng với seller không.** Câu lệnh dưới đây đếm ba thứ:
- số shop có `SKP_SELLER` không tìm thấy trong bảng seller;
- số shop nằm ở tỉnh khác với tỉnh của seller;
- số shop có model khác với model của seller.

In [ ]:
q('''select
 (select count(*) from salesroom s left join seller x using(SKP_SELLER) where x.SKP_SELLER is null) shop_co_seller_khong_ton_tai,
 (select count(*) from salesroom s join seller x using(SKP_SELLER) where s.SALESROOM_PROVINCE<>x.SELLER_PROVINCE) shop_khac_tinh_voi_seller,
 (select count(*) from salesroom s join seller x using(SKP_SELLER) where s.SALESROOM_MODEL<>x.SELLER_MODEL) shop_khac_model_voi_seller''')

,shop_co_seller_khong_ton_tai,shop_khac_tinh_voi_seller,shop_khac_model_voi_seller
0,15,1346,69


- **15 shop trỏ tới seller không có trong bảng seller.** Kết quả khi ghép bảng phụ thuộc vào cách ghép, xem ngay bên dưới.
- **1.346 shop nằm khác tỉnh với seller.** Chưa chắc là lỗi, vì một đối tác lớn có thể có shop ở nhiều tỉnh. Khi phân tích theo địa lý nên dùng tỉnh của shop.
- **69 shop có model khác với seller của nó.**

Để biết 15 shop này quan trọng đến đâu, câu lệnh dưới đây liệt kê chúng kèm trạng thái, ngành hàng, tỉnh, ngày xuất hiện trong dữ liệu (`SALESROOM_START`), và số hồ sơ, số lead, số lần phân công nhân viên đã có.

In [ ]:
q('''select s.SKP_SALESROOM, s.SKP_SELLER, s.SALESROOM_STATUS, s.SALESROOM_PRODUCT, s.SALESROOM_PROVINCE, s.SALESROOM_START,
  (select count(*) from contract c where c.skp_salesroom=s.SKP_SALESROOM::BIGINT) so_ho_so,
  (select count(*) from leads l where l.skp_salesroom=s.SKP_SALESROOM::BIGINT) so_lead,
  (select count(*) from e2s e where e.skp_salesroom=s.SKP_SALESROOM::BIGINT) so_phan_cong
from salesroom s left join seller x using(SKP_SELLER) where x.SKP_SELLER is null order by s.SALESROOM_START''')

,SKP_SALESROOM,SKP_SELLER,SALESROOM_STATUS,SALESROOM_PRODUCT,SALESROOM_PROVINCE,SALESROOM_START,so_ho_so,so_lead,so_phan_cong
0,126435301,43633638,Active,TW,Tây Ninh,17-10-2026 08:11:33,0,0,0
1,126429536,43303454,Active,CD,Gia Lai,17-10-2026 10:00:30,0,0,0
2,126548627,48059603,Active,CD,TP.HCM,17-10-2026 11:12:22,0,0,0
3,126518953,46416002,Active,CD,Lâm Đồng,17-10-2026 13:59:52,0,0,0
4,126487305,44624178,Active,TW,Hải Phòng,17-10-2026 16:46:21,0,0,0
5,126519108,46416006,Active,CD,Lâm Đồng,17-10-2026 17:24:52,0,0,0
6,126549079,48116651,Active,NS,Khánh Hòa,17-10-2026 18:08:36,0,0,0
7,126539183,47546376,Active,NS,Cà Mau,18-10-2026 10:16:36,0,0,0
8,126526906,46864177,Active,NS,Quảng Ninh,18-10-2026 10:19:06,0,0,0
9,126526893,46864177,Active,NS,Quảng Ninh,18-10-2026 10:19:07,0,0,0


Cả 15 shop đều xuất hiện trong dữ liệu vào ngày 17 hoặc 18/10/2026, tức sau khi dữ liệu contract đã kết thúc (30/09/2026). Chúng chưa có hồ sơ, lead hay phân công nhân viên nào. Nhiều khả năng đây là shop vừa được tạo, đối tác chưa kịp được đưa vào bảng seller.

Vì 15 shop này không có hoạt động nào để phân tích, nhóm **bỏ chúng khỏi dữ liệu sạch**. Số shop giảm từ 14.231 xuống 14.216, và không ảnh hưởng tới doanh số, lead hay số nhân viên.

**Kiểm tra 3.4. Một tỉnh có thuộc nhiều vùng không.** Nếu mỗi tỉnh chỉ thuộc một vùng, có thể suy ra vùng từ tỉnh. Câu lệnh dưới đây liệt kê những tỉnh có shop nằm ở từ 2 vùng (`SALESROOM_AREA`) trở lên.

In [ ]:
q('''select SALESROOM_PROVINCE, string_agg(distinct SALESROOM_AREA, ', ') cac_area, count(*) so_shop
from salesroom group by 1 having count(distinct SALESROOM_AREA)>1 order by 1''')

,SALESROOM_PROVINCE,cac_area,so_shop
0,Cần Thơ,"Mekong, Western",442
1,Hưng Yên,"North, North East",305
2,Lâm Đồng,"Eastern North, Eastern South",483
3,Phú Thọ,"North, North West",337
4,TP. Đồng Nai,"South East, Eastern South",720
5,TP.HCM,"South East, Sai Gon, Eastern South",2826
6,Tây Ninh,"Sai Gon, South East",400


Có 7 tỉnh thuộc nhiều vùng, ví dụ TP.HCM nằm ở 3 vùng: Sai Gon, Eastern South và South East.

Giả thuyết: vùng kinh doanh được chia theo **tỉnh cũ, trước sáp nhập**. Khi nhiều tỉnh cũ gộp thành một tỉnh mới, tỉnh mới đó trải qua nhiều vùng. Để kiểm tra, câu lệnh dưới đây lấy 622 shop còn ghi tên quận, huyện cũ (nên biết được shop nằm ở tỉnh cũ nào), thuộc 7 tỉnh có nhiều vùng. Với mỗi cặp tỉnh mới – vùng, nó liệt kê các quận, huyện cũ.

In [ ]:
q('''select SALESROOM_PROVINCE tinh_moi, SALESROOM_AREA vung, string_agg(distinct SALESROOM_DISTRICT, ', ') quan_huyen_cu, count(*) so_shop
from salesroom where SALESROOM_DISTRICT<>'.' and SALESROOM_PROVINCE in
  (select SALESROOM_PROVINCE from salesroom group by 1 having count(distinct SALESROOM_AREA)>1)
group by 1, 2 order by 1, 2''')

,tinh_moi,vung,quan_huyen_cu,so_shop
0,Cần Thơ,Mekong,"Q. Ninh Kiều, H. Phong Điền, Q. Ô Môn, H. Cờ Đỏ, H. Thới Lai, Q. Thốt Nốt",13
1,Cần Thơ,Western,"H. Châu Thành A, TP. Ngã Bảy, TP. Sóc Trăng, TP. Vị Thanh, H. Phụng Hiệp, H. Châu Thành",11
2,Hưng Yên,North,"H. Ân Thi, H. Văn Giang, H. Khoái Châu, H. Yên Mỹ, H. Văn Lâm",6
3,Hưng Yên,North East,"TP. Thái Bình, H. Vũ Thư, H. Quỳnh Phụ, H. Kiến Xương",8
4,Lâm Đồng,Eastern North,"H. Đức Trọng, H. Bảo Lâm, H. Lạc Dương, TP. Đà Lạt, H. Di Linh",7
5,Lâm Đồng,Eastern South,"H. Hàm Thuận Nam, TP. Phan Thiết, H. Đức Linh, TX. La Gi",9
6,Phú Thọ,North,"TP. Hòa Bình, H. Cao Phong, H. Mai Châu",3
7,Phú Thọ,North West,"H. Thanh Thủy, H. Cẩm Khê, TP. Việt Trì, H. Thanh Sơn",6
8,TP. Đồng Nai,Eastern South,"H. Nhơn Trạch, H. Vĩnh Cửu, TP. Long Khánh, H. Long Thành, H. Trảng Bom, TP. Biên Hòa, H. Cẩm Mỹ",21
9,TP.HCM,Eastern South,"TP. Bà Rịa, TP. Vũng Tàu, TP. Phú Mỹ, H. Châu Đức",13


Kết quả khớp với giả thuyết. Ví dụ:
- TP.HCM – Sai Gon gồm Q. 1, Q. Tân Bình, TP. Thủ Đức…, tức TP.HCM cũ. TP.HCM – South East gồm TP. Thủ Dầu Một, TP. Dĩ An, TP. Bến Cát…, tức Bình Dương cũ. TP.HCM – Eastern South gồm TP. Vũng Tàu, TP. Bà Rịa…, tức Bà Rịa – Vũng Tàu cũ.
- Tây Ninh – Sai Gon gồm H. Cần Giuộc, H. Tân Trụ, tức Long An cũ. Tây Ninh – South East là Tây Ninh cũ.
- Cần Thơ – Mekong là Cần Thơ cũ. Cần Thơ – Western gồm TP. Sóc Trăng, TP. Vị Thanh, tức Sóc Trăng và Hậu Giang cũ.
- Lâm Đồng – Eastern South gồm TP. Phan Thiết, TX. La Gi, tức Bình Thuận cũ.

Câu lệnh dưới đây kiểm tra thêm: có quận, huyện cũ nào bị chia vào nhiều hơn một vùng không. Nếu vùng đi theo tỉnh cũ thì con số này phải bằng 0.

In [ ]:
q('''select count(*) quan_huyen_cu_thuoc_nhieu_vung from (
  select SALESROOM_PROVINCE, SALESROOM_DISTRICT from salesroom where SALESROOM_DISTRICT<>'.'
  group by 1, 2 having count(distinct SALESROOM_AREA)>1)''')

,quan_huyen_cu_thuoc_nhieu_vung
0,0


Không có quận, huyện cũ nào nằm ở hai vùng. Vậy một tỉnh thuộc nhiều vùng **không phải lỗi**: `SALESROOM_AREA` là vùng kinh doanh công ty chia theo tỉnh cũ, còn `SALESROOM_PROVINCE` đã chuyển sang tỉnh mới. Hệ quả:
- không được suy ra vùng từ tỉnh, phải lấy đúng cột area của từng shop;
- khi chia shop theo khu vực, phải chọn giữa tỉnh mới và vùng; hai cột này cho kết quả khác nhau.

**Kiểm tra 3.5. Bao nhiêu shop còn ghi địa chỉ theo địa giới cũ.** Theo mô tả của Ban tổ chức, cột quận ghi dấu "." nghĩa là địa chỉ đã theo địa giới mới (không còn cấp quận). Ghi tên quận nghĩa là vẫn theo địa giới cũ. Câu lệnh dưới đây đếm số shop theo hai nhóm này.

In [ ]:
q('''select (SALESROOM_DISTRICT='.') ghi_theo_dia_gioi_moi, count(*) so_shop from salesroom group by 1''')

,ghi_theo_dia_gioi_moi,so_shop
0,True,13609
1,False,622


Có 622 shop (4,4%) vẫn ghi theo địa giới cũ. Cột quận chỉ cần khi phân tích ở cấp quận, huyện. Nếu nhóm phân tích ở cấp tỉnh và vùng thì không cần sửa cột này.

**Kiểm tra 3.6. Tên phường, xã có được ghi thống nhất không.** Câu lệnh đầu đếm số tên phường, xã khác nhau. Sau đó nó đếm lại sau khi chuẩn hóa: bỏ các tiền tố "P.", "Phường", "X.", "Xã", "TT.", "Thị trấn" (viết hoa hay thường đều được) và đưa về chữ thường. Riêng phường đánh số như "P. 6", "Phường 06" thì giữ dạng "phường 6", vì nếu chỉ còn con số thì không biết đó là phường. Nếu số giảm nhiều, nghĩa là cùng một nơi đang được ghi theo nhiều cách.

In [ ]:
TEN_CHUAN = '''case when regexp_matches(trim(regexp_replace(trim(SALESROOM_TOWN),'^(p\\.|phường|x\\.|xã|tt\\.|thị trấn)\\s*','','i')),'^[0-9]+$')
  then 'phường ' || cast(cast(trim(regexp_replace(trim(SALESROOM_TOWN),'^(p\\.|phường|x\\.|xã|tt\\.|thị trấn)\\s*','','i')) as int) as varchar)
  else lower(trim(regexp_replace(trim(SALESROOM_TOWN),'^(p\\.|phường|x\\.|xã|tt\\.|thị trấn)\\s*','','i'))) end'''
q(f'''select count(distinct SALESROOM_TOWN) so_ten_phuong_xa, count(distinct {TEN_CHUAN}) sau_khi_chuan_hoa from salesroom''')

,so_ten_phuong_xa,sau_khi_chuan_hoa
0,2315,2024


Số tên giảm từ 2.315 xuống 2.024. Để thấy cụ thể, câu lệnh dưới đây lấy vài nơi có cùng tên sau khi chuẩn hóa nhưng được ghi theo nhiều cách khác nhau. Các phường đánh số được xếp xuống cuối.

In [ ]:
q(f'''with t as (select distinct SALESROOM_TOWN, SALESROOM_PROVINCE, {TEN_CHUAN} ten_chuan from salesroom)
select SALESROOM_PROVINCE, ten_chuan, string_agg(SALESROOM_TOWN, '  |  ') cac_cach_ghi from t
group by 1, 2 having count(*)>1 order by (ten_chuan like 'phường %'), 1, 2 limit 10''')

,SALESROOM_PROVINCE,ten_chuan,cac_cach_ghi
0,An Giang,chợ mới,X. Chợ Mới | TT. Chợ Mới
1,An Giang,mỹ thới,P. Mỹ Thới | Phường Mỹ Thới
2,An Giang,mỹ xuyên,Phường Mỹ Xuyên | P. MỸ XUYÊN
3,An Giang,phú hòa,TT. PHÚ HÒA | X. Phú Hòa
4,Bắc Giang,nếnh,Phường Nếnh | P. NẾNH
5,Bắc Ninh,nam sơn,Phường Nam Sơn | P. Nam Sơn
6,Bắc Ninh,đào viên,P. Đào Viên | X. Đào Viên
7,Cao Bằng,quảng uyên,X. Quảng Uyên | Thị Trấn Quảng Uyên
8,Cà Mau,cái đôi vàm,X. Cái Đôi Vàm | Thị Trấn Cái Đôi Vàm
9,Cà Mau,hưng mỹ,XÃ HƯNG MỸ | X. Hưng Mỹ


Cùng một phường được ghi theo nhiều cách:
- "P. Mỹ Thới" và "Phường Mỹ Thới";
- "Phường Mỹ Xuyên" và "P. MỸ XUYÊN" (viết hoa toàn bộ);
- "XÃ HƯNG MỸ" và "X. Hưng Mỹ".

Muốn ghép dữ liệu cấp phường, xã thì phải chuẩn hóa tên trước. Có hai lưu ý:
- Có trường hợp như "X. Chợ Mới" và "TT. Chợ Mới", hay "X. Quảng Uyên" và "Thị Trấn Quảng Uyên", có thể là hai đơn vị khác nhau (xã mới và thị trấn cũ), nên không được gộp máy móc.
- Phường đánh số như "phường 6" là phường cũ. Cùng một số có thể nằm ở nhiều quận khác nhau trong cùng tỉnh, nên khi ghép phải đi kèm tên quận.

**Kiểm tra 3.7. `SALESROOM_START` có phải ngày mở shop thật không.** Ở Kiểm tra 1.1, ngày bắt đầu sớm nhất của shop là 02/01/2025, trong khi seller có từ năm 2008. Câu lệnh đầu đếm số shop theo quý bắt đầu. Câu lệnh sau đếm số contract được nộp tại shop trước ngày bắt đầu của chính shop đó.

In [ ]:
q('''select strftime(date_trunc('quarter',strptime(SALESROOM_START,'%d-%m-%Y %H:%M:%S')),'%Y-%m') quy_bat_dau, count(*) so_shop
from salesroom group by 1 order by 1''')

,quy_bat_dau,so_shop
0,2025-01,1323
1,2025-04,2402
2,2025-07,2009
3,2025-10,2215
4,2026-01,1598
5,2026-04,2478
6,2026-07,1912
7,2026-10,294


In [ ]:
q('''select count(*) contract_tai_shop_trong_bang,
  sum((c.date_apply < strptime(s.SALESROOM_START,'%d-%m-%Y %H:%M:%S'))::int) nop_truoc_ngay_start
from contract c join salesroom s on c.skp_salesroom=s.SKP_SALESROOM::BIGINT''')

,contract_tai_shop_trong_bang,nop_truoc_ngay_start
0,621387,0.00


Cả 14.231 shop đều bắt đầu từ quý 1/2025 trở đi, rải đều qua các quý. Không có contract nào được nộp trước ngày bắt đầu của shop. Có hai cách hiểu:
- `SALESROOM_START` là ngày shop được đưa vào hệ thống hiện tại, không phải ngày shop mở cửa;
- hoặc bảng salesroom chỉ gồm các shop bắt đầu từ năm 2025.

Hệ quả: "số tháng shop đã hoạt động" chỉ tính được từ ngày này.

## Phần 4. Bảng emp_2_salesroom (phân công nhân viên)

**Kiểm tra 4.1. Có dòng trùng hoặc khoảng thời gian sai không.** Câu lệnh dưới đây đếm:
- số dòng trùng hoàn toàn (mọi cột giống nhau);
- số dòng có ngày kết thúc trước ngày bắt đầu (`to_truoc_from`);
- số dòng có ngày kết thúc bằng đúng ngày bắt đầu (`khoang_bang_0`).

In [ ]:
q('''select count(*) so_dong, (select count(*) from (select distinct * from e2s)) so_dong_khac_nhau,
  count(*) - (select count(*) from (select distinct * from e2s)) dong_trung_hoan_toan,
  sum((dtime_valid_to<dtime_valid_from)::int) to_truoc_from, sum((dtime_valid_to=dtime_valid_from)::int) khoang_bang_0
from e2s''')

,so_dong,so_dong_khac_nhau,dong_trung_hoan_toan,to_truoc_from,khoang_bang_0
0,275856,274944,912,0.00,20.00


Có 912 dòng trùng hoàn toàn, cần bỏ trước khi đếm. Có 20 lần phân công dài đúng 0 giây. Không có dòng nào có ngày kết thúc trước ngày bắt đầu.

**Kiểm tra 4.2. Ngày kết thúc 01/01/3000 nghĩa là gì.** Ở Round 2, StructureTable của Ban tổ chức đã dùng ngày 3000-01-01 làm **giá trị quy ước**: với lead từ quy trình SOB, nhân viên hỗ trợ khách trực tiếp mà không bấm nhận lead, nên ngày nhận lead được ghi là 3000-01-01. Nghĩa là hệ thống dùng ngày này để đánh dấu một bước chưa xảy ra hoặc không áp dụng, đây không phải lỗi nhập liệu. Câu lệnh dưới đây kiểm tra xem ngày 01/01/3000 ở cột `dtime_valid_to` đi kèm trạng thái phân công nào.

In [ ]:
q('''select code_status, year(dtime_valid_to)=3000 as den_nam_3000, count(*) so_dong from e2s group by all order by all''')

,code_status,den_nam_3000,so_dong
0,a,True,233807
1,n,False,42049


Mọi phân công có `code_status = a` đều có ngày kết thúc 01/01/3000. Mọi phân công `code_status = n` đều có ngày kết thúc thật. Vậy ở bảng này, 01/01/3000 nghĩa là **phân công chưa kết thúc, nhân viên vẫn đang được giao ở shop đó**.

Khi tính thời gian nhân viên làm ở shop, cần thay ngày này bằng ngày cuối của dữ liệu (17/10/2026). Nếu không, thời gian làm việc sẽ ra gần 1.000 năm.

**Kiểm tra 4.3. Bảng phân công có khớp với người làm hồ sơ không.** Câu lệnh dưới đây lấy từng hồ sơ CD/TW tại các shop có trong bảng salesroom. Với mỗi hồ sơ, nó kiểm tra xem nhân viên làm hồ sơ có đang được phân công tại đúng shop đó vào lúc khách nộp hồ sơ hay không.

In [ ]:
q('''with c as (select c.skp_credit_case, c.skp_salesroom, c.skp_emp_pre_process::BIGINT emp, c.date_apply
            from contract c join salesroom s on c.skp_salesroom=s.SKP_SALESROOM::BIGINT where c.product in ('CD','TW')),
m as (select c.skp_credit_case, max((e.skp_employee is not null)::int) co_phan_cong
      from c left join e2s e on e.skp_employee=c.emp and e.skp_salesroom=c.skp_salesroom and c.date_apply between e.dtime_valid_from and e.dtime_valid_to group by 1)
select count(*) ho_so_cd_tw, sum(co_phan_cong) nhan_vien_co_phan_cong_luc_nop from m''')

,ho_so_cd_tw,nhan_vien_co_phan_cong_luc_nop
0,524085,"491,937.00"


Có 491.937 trên 524.085 hồ sơ (94%) do nhân viên đang được phân công tại đúng shop đó làm. Hai bảng khớp nhau ở mức chấp nhận được, nên có thể dùng bảng phân công để kiểm tra chéo.

## Phần 5. Bảng contract

**Kiểm tra 5.1. Mã hồ sơ có trùng không, cột nào bị trống.** Câu lệnh dưới đây so số dòng với số mã `skp_credit_case` khác nhau, và đếm số ô trống của các cột chính.

In [ ]:
q('''select count(*) so_dong, count(distinct skp_credit_case) so_ma_khac_nhau,
  count(*)-count(skp_salesroom) trong_salesroom, count(*)-count(product) trong_product, count(*)-count(skp_client) trong_client,
  count(*)-count(credit_amount) trong_credit, count(*)-count(price) trong_price, count(*)-count(payment_num) trong_payment_num
from contract''')

,so_dong,so_ma_khac_nhau,trong_salesroom,trong_product,trong_client,trong_credit,trong_price,trong_payment_num
0,3775371,3775371,55,49147,7,7,2995428,1417426


Mã hồ sơ không trùng. Có 4 cột bị trống nhiều: `product` (49.147), `price` (gần 3 triệu), `payment_num` (1,4 triệu) và một ít ở `skp_client`, `credit_amount` (7). Các kiểm tra dưới đây xem từng chỗ trống có phải lỗi không.

**Kiểm tra 5.2. Hồ sơ trống `product` là hồ sơ gì.** Câu lệnh dưới đây đếm các hồ sơ không có product theo trạng thái.

In [ ]:
q('''select name_status, count(*) so_ho_so from contract where product is null group by 1 order by 2 desc''')

,name_status,so_ho_so
0,Rejected,37452
1,Cancelled,11688
2,In Preprocess,4
3,In Process,3


Mọi hồ sơ không có product đều ở trạng thái bị từ chối, bị hủy hoặc đang xử lý, nên không ảnh hưởng doanh số.

**Kiểm tra 5.3. 7 hồ sơ trống `skp_client` trông như thế nào.** Câu lệnh dưới đây in ra các hồ sơ này.

In [ ]:
q('''select * from contract where skp_client is null limit 7''')

,skp_salesroom,skp_credit_case,skp_emp_pre_process,skp_product,product,name_status,price,init_pay,credit_amount,ins_amount,payment_num,skp_client,date_sign,date_apply,amt_billing_1st,amt_billing_2nd,amt_billing_3rd,dpd_1pd,dpd_2pd,dpd_3pd,dpd_4pd
0,5575,361409504,"17,739,927.00",NaN,None,In Process,NaN,NaN,NaN,NaN,NaN,NaN,3000-01-01,2026-05-23 21:08:49,NaN,NaN,NaN,0.00,0.00,0.00,0.00
1,25855332,363759926,"21,929,162.00",NaN,None,In Preprocess,NaN,NaN,NaN,NaN,NaN,NaN,3000-01-01,2026-07-01 08:32:59,NaN,NaN,NaN,0.00,0.00,0.00,0.00
2,57669812,342293393,NaN,NaN,None,In Process,NaN,NaN,NaN,NaN,NaN,NaN,3000-01-01,2025-05-21 12:37:18,NaN,NaN,NaN,0.00,0.00,0.00,0.00
3,25856388,337963924,"15,875,406.00",NaN,None,In Preprocess,NaN,NaN,NaN,NaN,NaN,NaN,3000-01-01,2025-02-23 09:46:26,NaN,NaN,NaN,0.00,0.00,0.00,0.00
4,42902170,354198995,"12,897,653.00",NaN,None,In Preprocess,NaN,NaN,NaN,NaN,NaN,NaN,3000-01-01,2026-01-14 14:46:32,NaN,NaN,NaN,0.00,0.00,0.00,0.00
5,2878390,339760634,"17,732,584.00",NaN,None,In Process,NaN,NaN,NaN,NaN,NaN,NaN,3000-01-01,2025-04-02 11:22:46,NaN,NaN,NaN,0.00,0.00,0.00,0.00
6,25882297,337955257,"13,155,469.00",NaN,None,In Preprocess,NaN,NaN,NaN,NaN,NaN,NaN,3000-01-01,2025-02-23 09:27:59,NaN,NaN,NaN,0.00,0.00,0.00,0.00


Cả 7 hồ sơ đều đang xử lý (In Process hoặc In Preprocess) và thiếu gần hết thông tin. Có thể bỏ.

**Kiểm tra 5.4. `price` và `payment_num` trống ở sản phẩm nào.** Theo mô tả, chỉ hồ sơ mua hàng (CD, TW) mới có giá hàng, và sản phẩm hạn mức (BNPL, CC) không chia kỳ cố định. Câu lệnh dưới đây đếm số ô trống của hai cột theo từng sản phẩm để kiểm tra điều này.

In [ ]:
q('''select coalesce(product,'(trống)') product, count(*) so_ho_so, count(*)-count(price) trong_price, count(*)-count(payment_num) trong_payment_num
from contract group by 1 order by 1''')

,product,so_ho_so,trong_price,trong_payment_num
0,(trống),49147,49147,49147
1,BNPL,951483,951483,951483
2,CC,416796,416796,416796
3,CD,450608,0,0
4,CL,1389963,1389963,0
5,CW,188039,188039,0
6,TW,329335,0,0


Đúng như thiết kế:
- `price` chỉ có ở CD và TW, trống ở mọi sản phẩm khác.
- `payment_num` trống ở BNPL, CC và các hồ sơ không có product.

Đây không phải lỗi.

**Kiểm tra 5.5. Ngày ký 01/01/3000 đi kèm trạng thái nào.** Theo StructureTable của Round 2, hồ sơ được tính là đã ký khi có trạng thái Active, Finished, Paid off, Signed, Written off; các trạng thái khác là chưa ký. Câu lệnh dưới đây đếm, với mỗi trạng thái, có bao nhiêu hồ sơ có ngày ký 01/01/3000, kèm ngày ký sớm nhất và muộn nhất.

In [ ]:
q('''select name_status, count(*) so_ho_so, sum((year(date_sign)=3000)::int) ngay_ky_nam_3000,
  min(date_sign) som_nhat, max(date_sign) muon_nhat from contract group by 1 order by 2 desc''')

,name_status,so_ho_so,ngay_ky_nam_3000,som_nhat,muon_nhat
0,Active,1407088,0.00,2025-01-01 01:10:16,2026-10-14 17:07:51
1,Finished,1221992,0.00,2025-01-01 00:38:31,2026-10-11 16:09:11
2,Rejected,565776,"565,776.00",3000-01-01 00:00:00,3000-01-01 00:00:00
3,Cancelled,493600,"483,085.00",2025-01-01 08:41:13,3000-01-01 00:00:00
4,Written off,62957,0.00,2025-01-01 04:07:44,2026-06-13 20:34:34
5,Paid off,23827,0.00,2025-01-01 08:06:05,2026-09-20 16:50:13
6,Signed,121,0.00,2025-05-08 17:08:30,2026-10-09 18:41:58
7,In Preprocess,4,4.00,3000-01-01 00:00:00,3000-01-01 00:00:00
8,Approved,3,3.00,3000-01-01 00:00:00,3000-01-01 00:00:00
9,In Process,3,3.00,3000-01-01 00:00:00,3000-01-01 00:00:00


Kết quả khớp với quy ước "chưa ký":
- Mọi hồ sơ Rejected, In Process, In Preprocess và Approved có ngày ký 01/01/3000.
- Mọi hồ sơ đã ký (Active, Finished, Paid off, Written off, Signed) có ngày ký thật.
- Cancelled có 483.085 hồ sơ với ngày ký 01/01/3000, tức bị hủy trước khi ký. Còn 10.515 hồ sơ có ngày ký thật, tức đã ký rồi mới bị hủy.

Vậy ở cột `date_sign`, 01/01/3000 là **giá trị quy ước nghĩa là hồ sơ chưa đi tới bước ký**, giống cách Round 2 dùng ngày 3000-01-01. Đây không phải lỗi. Khi tính các chỉ số theo ngày ký (ví dụ số hợp đồng ký mỗi tháng, thời gian từ nộp tới ký), phải loại các dòng này. Nếu không, năm 3000 sẽ xuất hiện như một tháng có hơn nửa triệu hồ sơ.

**Kiểm tra 5.6. Số tiền vay của từng sản phẩm nằm ở cột nào.** Câu lệnh dưới đây lấy các hợp đồng đã ký, rồi đếm theo sản phẩm: bao nhiêu hợp đồng có `credit_amount` bằng 0, và bao nhiêu hợp đồng có số tiền ở cột bảng kê tháng đầu (`amt_billing_1st`).

In [ ]:
q(f'''select product, count(*) da_ky, sum((credit_amount=0)::int) credit_bang_0, count(amt_billing_1st) co_billing_ky_1
from contract where name_status in {SIGNED} group by 1 order by 1''')

,product,da_ky,credit_bang_0,co_billing_ky_1
0,BNPL,826402,"584,049.00",531162
1,CC,310351,"310,351.00",284679
2,CD,289092,0.00,0
3,CL,937469,0.00,0
4,CW,70195,0.00,0
5,TW,282476,0.00,0


- Với CC, `credit_amount` luôn bằng 0.
- Với BNPL, 584.049 trên 826.402 hợp đồng đã ký có `credit_amount` bằng 0.
- Cột `amt_billing_1st` chỉ có ở CC và BNPL.

Nếu tính **funded volume** (tổng tiền cho khách vay) bằng cách cộng `credit_amount`, thì CC và BNPL gần như không đóng góp gì. Với hai sản phẩm này phải dùng cột `amt_billing`, tức số tiền khách thực sự tiêu theo bảng kê. Lưu ý: cột billing cũng chỉ có ở khoảng 56–68% hợp đồng CC và BNPL.

**Kiểm tra 5.7. Cột DPD có những giá trị nào.** StructureTable mô tả DPD là "số ngày quá hạn". Câu lệnh dưới đây liệt kê các giá trị khác nhau của 4 cột DPD và đếm số dòng cho mỗi giá trị.

In [ ]:
q('''select 'dpd_1pd' cot, dpd_1pd gia_tri, count(*) so_dong from contract group by 2
union all select 'dpd_2pd', dpd_2pd, count(*) from contract group by 2
union all select 'dpd_3pd', dpd_3pd, count(*) from contract group by 2
union all select 'dpd_4pd', dpd_4pd, count(*) from contract group by 2 order by 1, 2''')

,cot,gia_tri,so_dong
0,dpd_1pd,0.00,3593129
1,dpd_1pd,1.00,182242
2,dpd_2pd,0.00,3538220
3,dpd_2pd,1.00,237151
4,dpd_3pd,0.00,3531058
5,dpd_3pd,1.00,244313
6,dpd_4pd,0.00,3559060
7,dpd_4pd,1.00,216311


Cả 4 cột chỉ có hai giá trị 0 và 1. Vậy DPD thực chất là cờ "có trễ hạn ở kỳ đó hay không", không phải số ngày.

**Kiểm tra 5.8. Tỷ lệ trễ hạn có đều theo tháng ký không.** Câu lệnh dưới đây lấy các hợp đồng đã ký, nhóm theo tháng ký, và tính phần trăm hợp đồng bị trễ ở kỳ 1, 2, 3, 4.

In [ ]:
q(f'''select strftime(date_trunc('month',date_sign),'%Y-%m') thang_ky, count(*) so_hd,
  round(avg(dpd_1pd)*100,2) pct_tre_ky1, round(avg(dpd_2pd)*100,2) pct_tre_ky2, round(avg(dpd_3pd)*100,2) pct_tre_ky3, round(avg(dpd_4pd)*100,2) pct_tre_ky4
from contract where name_status in {SIGNED} group by 1 order by 1''')

,thang_ky,so_hd,pct_tre_ky1,pct_tre_ky2,pct_tre_ky3,pct_tre_ky4
0,2025-01,145794,0.21,2.35,9.31,7.91
1,2025-02,111468,1.69,8.08,8.86,7.12
2,2025-03,133258,7.06,8.96,9.09,7.73
3,2025-04,134174,6.82,8.36,9.47,7.53
4,2025-05,131129,6.76,9.32,9.52,7.96
5,2025-06,138976,7.44,9.15,9.69,8.11
6,2025-07,134905,7.35,9.48,9.94,8.12
7,2025-08,129593,7.65,10.29,10.06,8.49
8,2025-09,128273,7.34,9.58,9.75,10.00
9,2025-10,134899,7.22,9.29,9.94,10.53


Có hai chỗ lệch theo thời gian:

- **Hợp đồng ký gần đây chưa đến kỳ trả nên luôn bằng 0.**
  - Hợp đồng ký tháng 08/2026 có tỷ lệ trễ kỳ 3 và kỳ 4 bằng 0.
  - Hợp đồng ký tháng 09/2026 có tỷ lệ trễ kỳ 1 chỉ 4,2%, trong khi các tháng trước khoảng 7–8%.
  - Đây là hiện tượng **right-censoring** (dữ liệu bị cắt ở cuối kỳ quan sát): con số 0 ở đây nghĩa là "chưa đến hạn", không phải "trả đúng hạn".
  - Nếu tính rủi ro trên mọi hợp đồng, shop mới hoặc shop đang tăng nhanh sẽ trông an toàn hơn thực tế. Khi tính tỷ lệ trễ kỳ 1, chỉ nên lấy hợp đồng ký trước 08/2026.
- **Hợp đồng ký đầu năm 2025 có tỷ lệ trễ kỳ 1 thấp bất thường.**
  - Tháng 01/2025 chỉ 0,21%, tháng 02/2025 là 1,69%, trong khi từ tháng 03/2025 trở đi khoảng 7%.
  - Nhiều khả năng trễ hạn chỉ bắt đầu được ghi nhận từ khoảng tháng 03/2025. Không nên đưa hai tháng này vào phép so sánh rủi ro.

**Kiểm tra 5.9. Hợp đồng có nằm ở shop trong bảng salesroom không.** Câu lệnh dưới đây đếm, theo từng sản phẩm, bao nhiêu hồ sơ có `skp_salesroom` tìm thấy trong bảng salesroom.

In [ ]:
q('''select coalesce(c.product,'(trống)') product, count(*) so_ho_so, sum((s.SKP_SALESROOM is not null)::int) tai_shop_trong_bang,
  round(100.0*sum((s.SKP_SALESROOM is not null)::int)/count(*),1) pct_trong_bang
from contract c left join salesroom s on c.skp_salesroom=s.SKP_SALESROOM::BIGINT group by 1 order by 2 desc''')

,product,so_ho_so,tai_shop_trong_bang,pct_trong_bang
0,CL,1389963,"24,753.00",1.80
1,BNPL,951483,"43,892.00",4.60
2,CD,450608,"450,608.00",100.00
3,CC,416796,"18,198.00",4.40
4,TW,329335,"73,477.00",22.30
5,CW,188039,"2,046.00",1.10
6,(trống),49147,"8,413.00",17.10


Toàn bộ hồ sơ CD nằm ở shop có trong bảng, nhưng TW chỉ có 22,3%. Các sản phẩm vay tiền mặt, thẻ và BNPL gần như không nằm ở shop trong bảng, đúng như đề nói các sản phẩm này đi qua kênh online.

TW thì khác: đây là hàng mua tại shop, đáng lẽ phải nằm ở shop. Câu lệnh dưới đây đếm:
- số mã điểm bán có hồ sơ TW nhưng không có trong bảng salesroom, và số hồ sơ TW ở các mã đó;
- trong các mã đó, bao nhiêu mã còn có hồ sơ trong quý 3/2026;
- số shop TW có trong bảng salesroom, để so sánh.

In [ ]:
q('''with x as (select c.skp_salesroom, count(*) n, max(c.date_apply) lan_cuoi from contract c left join salesroom s on c.skp_salesroom=s.SKP_SALESROOM::BIGINT
            where s.SKP_SALESROOM is null and c.product='TW' group by 1)
select count(*) ma_diem_ban_tw_ngoai_bang, sum(n) so_ho_so, sum((lan_cuoi>='2026-07-01')::int) con_ban_trong_quy3_2026,
  (select count(*) from salesroom where SALESROOM_PRODUCT='TW') shop_tw_trong_bang from x''')

,ma_diem_ban_tw_ngoai_bang,so_ho_so,con_ban_trong_quy3_2026,shop_tw_trong_bang
0,2918,"255,858.00","1,972.00",2631


Có 2.918 mã điểm bán TW không có trong bảng salesroom, chứa 255.858 hồ sơ, và 1.972 mã vẫn còn bán trong quý 3/2026. Trong khi đó bảng salesroom chỉ có 2.631 shop TW. Như vậy bảng salesroom đang thiếu phần lớn shop xe máy đang hoạt động.

Hệ quả: kết luận về shop TW chỉ đại diện cho một phần mạng lưới xe máy. Cần ghi rõ giới hạn này trong slide.

**Kiểm tra 5.10. Doanh số tăng vì shop bán nhiều hơn, hay vì có thêm shop.** Câu lệnh dưới đây lấy hồ sơ tại các shop có trong bảng. Với mỗi tháng, nó đếm số shop có hồ sơ, tổng số hồ sơ, và số hồ sơ trung bình mỗi shop.

In [ ]:
q('''select strftime(date_trunc('month',c.date_apply),'%Y-%m') thang, count(distinct c.skp_salesroom) so_shop_co_ho_so, count(*) so_ho_so,
  round(count(*)/count(distinct c.skp_salesroom),1) ho_so_moi_shop
from contract c join salesroom s on c.skp_salesroom=s.SKP_SALESROOM::BIGINT group by 1 order by 1''')

,thang,so_shop_co_ho_so,so_ho_so,ho_so_moi_shop
0,2025-01,189,1526,8.10
1,2025-02,325,3037,9.30
2,2025-03,730,6329,8.70
3,2025-04,1065,9331,8.80
4,2025-05,1393,11659,8.40
5,2025-06,1945,15819,8.10
6,2025-07,2511,21103,8.40
7,2025-08,2824,23940,8.50
8,2025-09,3057,26860,8.80
9,2025-10,3397,31368,9.20


Tổng số hồ sơ tăng từ 1.526 (01/2025) lên 55.743 (09/2026), gấp khoảng 37 lần. Nhưng số hồ sơ trung bình mỗi shop gần như đứng yên ở 8–9 hồ sơ mỗi tháng. Mức tăng tổng gần như hoàn toàn đến từ số shop có hồ sơ tăng, từ 189 lên 6.547.

Hệ quả cho bước tính BAU (mức nếu để yên): phải dự báo cho từng shop. Không được lấy xu hướng tăng của toàn mạng lưới áp cho từng shop.

**Kiểm tra 5.11. Một shop bán được bao nhiêu hợp đồng mỗi tháng.** Câu lệnh dưới đây lấy các hợp đồng đã ký trong quý 3/2026 (07–09/2026), tính số hợp đồng trung bình mỗi tháng cho từng shop, rồi xem phân bố: mức 25%, trung vị, mức 75%, và số shop dưới 2 hợp đồng mỗi tháng.

In [ ]:
q(f'''with x as (select c.skp_salesroom, count(*)/3.0 tb_thang from contract c join salesroom s on c.skp_salesroom=s.SKP_SALESROOM::BIGINT
            where c.date_apply>='2026-07-01' and c.name_status in {SIGNED} group by 1)
select count(*) so_shop_co_hop_dong_q3, quantile_cont(tb_thang,0.25) p25, quantile_cont(tb_thang,0.5) trung_vi, quantile_cont(tb_thang,0.75) p75,
  sum((tb_thang<2)::int) duoi_2_hd_moi_thang, (select count(*) from salesroom where SALESROOM_STATUS='Active') shop_active from x''')

,so_shop_co_hop_dong_q3,p25,trung_vi,p75,duoi_2_hd_moi_thang,shop_active
0,7886,0.67,2.00,5.00,"3,755.00",11847


Chỉ 7.886 shop có ít nhất một hợp đồng ký trong quý 3/2026, trong khi có 11.847 shop Active. Trung vị chỉ 2 hợp đồng mỗi tháng, và 3.755 shop bán dưới 2 hợp đồng mỗi tháng.

Với số nhỏ như vậy, chỉ cần một hợp đồng hơn hay kém cũng làm tỷ lệ của shop thay đổi rất mạnh. Đây là lý do phải đánh giá độ tin cậy của con số cho từng shop.

## Phần 6. Bảng contract_product

**Kiểm tra 6.1. Mỗi hồ sơ có đúng một dòng không.** Câu lệnh dưới đây so số dòng với số hồ sơ khác nhau, đếm số dòng trùng hoàn toàn, và đếm số ô trống ở cột `goods_group`.

In [ ]:
q('''select count(*) so_dong, count(distinct skp_credit_case) so_ho_so, (select count(*) from (select distinct * from cp)) dong_khac_nhau,
  count(*) - (select count(*) from (select distinct * from cp)) dong_trung_hoan_toan,
  count(*)-count(goods_group) trong_goods_group from cp''')

,so_dong,so_ho_so,dong_khac_nhau,dong_trung_hoan_toan,trong_goods_group
0,784902,779941,783502,1400,12476


Bảng có 784.902 dòng nhưng chỉ 779.941 hồ sơ, tức dư 4.961 dòng. Trong đó 1.400 dòng trùng hoàn toàn.

**Kiểm tra 6.2. Các hồ sơ nhiều dòng có nội dung giống hay khác nhau.** Câu lệnh dưới đây lấy các hồ sơ có từ 2 dòng trở lên, rồi đếm bao nhiêu hồ sơ có mọi dòng giống hệt nhau và bao nhiêu hồ sơ có dòng khác nhau.

In [ ]:
q('''with d as (select skp_credit_case, count(distinct (name_producer,name_goods_category,name_goods_type,coalesce(goods_group,''))) nd from cp group by 1 having count(*)>1)
select count(*) ho_so_nhieu_dong, sum((nd=1)::int) noi_dung_giong, sum((nd>1)::int) noi_dung_khac from d''')

,ho_so_nhieu_dong,noi_dung_giong,noi_dung_khac
0,3953,875.00,"3,078.00"


Có 3.953 hồ sơ có nhiều dòng: 875 hồ sơ có các dòng giống hệt nhau, 3.078 hồ sơ có các dòng khác nhau. Câu lệnh dưới đây in ví dụ 2 hồ sơ có dòng khác nhau.

In [ ]:
q('''with d as (select skp_credit_case from cp group by 1 having count(distinct name_goods_type)>1 limit 2) select cp.* from cp join d using(skp_credit_case) order by 1''')

,skp_credit_case,name_producer,name_goods_category,name_goods_type,goods_group
0,350611767,HONDA,Two weelers,Two weelers,Two Wheelers
1,350611767,HONDA,Two weelers,Other Two Wheelers,Two Wheelers
2,350643747,HONDA,Two weelers,Two weelers,Two Wheelers
3,350643747,HONDA,Two weelers,Other Two Wheelers,Two Wheelers


Cùng một hồ sơ vừa ghi loại hàng "Two weelers" vừa ghi "Other Two Wheelers". Nếu nối thẳng bảng này với contract, số hợp đồng sẽ bị đếm dư. Cần giữ 1 dòng cho mỗi hồ sơ trước khi nối.

**Kiểm tra 6.3. Tên nhóm hàng có được ghi thống nhất không.** Câu lệnh dưới đây liệt kê các giá trị `goods_group` có vẻ trùng nghĩa nhưng viết khác nhau, và các dòng trống `goods_group`, kèm `name_goods_category` để biết đó là hàng gì.

In [ ]:
q('''select goods_group, name_goods_category, count(*) so_dong from cp
where goods_group is null or upper(goods_group) in ('FITNESS','CONSTRUCTION','OTHER','OTHERS','FURNITURE','FURNITURE & MATTRESSES')
group by all order by 1, 2''')

,goods_group,name_goods_category,so_dong
0,CONSTRUCTION,HOUSE IMPROVEMENTS,113
1,Construction,HOUSE IMPROVEMENTS,54
2,Construction,HOUSE_EQUIPMENTS,13
3,FITNESS,FITNESS,23995
4,FURNITURE & MATTRESSES,FURNITURE & MATTRESSES,548
5,Fitness,FITNESS,21163
6,Furniture,FURNITURE & MATTRESSES,24800
7,Furniture,Furniture,266
8,OTHER,CAR EQUIPMENT,2
9,Others,CD SECOND HAND,15025


- **Cùng nhóm hàng nhưng viết khác nhau.** Ví dụ "FITNESS" và "Fitness", "CONSTRUCTION" và "Construction", "OTHER" và "Others", "FURNITURE & MATTRESSES" và "Furniture". Cần gộp về một tên.
- **Thiếu nhóm hàng.** 12.476 dòng trống `goods_group`. Phần lớn là xe máy ("Two weelers": 9.541 dòng, "USED MOTORBIKE": 1.228 dòng), nên điền lại được từ `name_goods_category`.
- **Lỗi chính tả.** "Two weelers" viết thiếu chữ h. Không ảnh hưởng tính toán, nhưng nên sửa khi đưa lên slide.

## Phần 7. Bảng leads

**Kiểm tra 7.1. Mã lead `id_uuid` có duy nhất không.** Câu lệnh dưới đây so số dòng với số mã khác nhau.

In [ ]:
q('''select count(*) so_dong, count(distinct id_uuid) so_uuid_khac_nhau, count(*)-count(distinct id_uuid) dong_du from leads''')

,so_dong,so_uuid_khac_nhau,dong_du
0,6130637,5921646,208991


Mã lead bị lặp: có 208.991 dòng dư. Để hiểu vì sao, câu lệnh dưới đây in ví dụ vài mã lead bị lặp, xếp theo thời gian tạo.

In [ ]:
q('''with d as (select id_uuid from leads group by 1 having count(*)>1)
select l.id_uuid, l.dtime_created, l.date_assigned, l.code_lead_status, l.result, l.code_product_type_original, l.skp_client
from leads l semi join d using(id_uuid) order by l.id_uuid, l.dtime_created limit 6''')

,id_uuid,dtime_created,date_assigned,code_lead_status,result,code_product_type_original,skp_client
0,00001418-efe0-4930-972e-56cc8c7c7f29,2026-01-26 14:47:51,2026-01-27 08:00:51,4,1.00,CCX,4997731
1,00001418-efe0-4930-972e-56cc8c7c7f29,2026-01-26 14:47:56,2026-01-27 08:01:21,4,1.00,CLX,4997731
2,00008a2f-4a86-4a52-be8e-b02cb2b3c476,2026-01-31 19:29:30,2026-02-01 08:01:00,4,1.00,CLX,70227945
3,00008a2f-4a86-4a52-be8e-b02cb2b3c476,2026-01-31 19:29:31,2026-02-01 08:01:09,4,1.00,CCX,70227945
4,0000e4ae-aa26-45f4-849c-4bed40c2d84f,2026-02-05 11:11:02,2026-02-05 17:42:36,4,2.00,CLX,60249786
5,0000e4ae-aa26-45f4-849c-4bed40c2d84f,2026-02-05 11:11:03,2026-02-05 17:42:36,4,2.00,CCX,60249786


Trong các ví dụ, cùng một mã lead, cùng một khách, được tạo cách nhau vài giây với hai sản phẩm khác nhau: CCX (thẻ) và CLX (vay tiền mặt).

Câu lệnh dưới đây kiểm tra trên toàn bộ các cặp dòng trùng mã: hai dòng trong cặp khác nhau ở sản phẩm, giờ tạo, kết quả, hay khách.

In [ ]:
q('''with d as (select id_uuid from leads group by 1 having count(*)>1),
x as (select l.*, row_number() over (partition by l.id_uuid order by l.dtime_created, l.date_assigned) rn from leads l semi join d using(id_uuid))
select count(*) cap_trung, sum((a.code_product_type_original<>b.code_product_type_original)::int) khac_san_pham,
  sum((a.dtime_created<>b.dtime_created)::int) khac_gio_tao, sum((a.result is distinct from b.result)::int) khac_ket_qua,
  sum((a.skp_client<>b.skp_client)::int) khac_khach
from x a join x b on a.id_uuid=b.id_uuid and a.rn=1 and b.rn=2''')

,cap_trung,khac_san_pham,khac_gio_tao,khac_ket_qua,khac_khach
0,208990,"102,973.00","199,308.00","59,430.00",0.00


Có 208.990 cặp:
- Hai dòng trong cặp **luôn cùng một khách**.
- Khoảng một nửa số cặp (102.973) khác nhau ở sản phẩm.
- Phần lớn các cặp (199.308) khác nhau ở giờ tạo.

Vậy mỗi dòng nên được hiểu là "một lần chào một sản phẩm cho một khách". Không được dùng `id_uuid` làm khóa duy nhất, và không được bỏ dòng trùng theo `id_uuid`.

**Kiểm tra 7.2. Các cột nào bị trống.** Câu lệnh dưới đây đếm:
- số ô trống ở `skp_employee`, `amt_loan_limit` và `result`;
- số dòng ghi XNA (không có thông tin) ở `code_sa` và `text_accepted_by`.

In [ ]:
q('''select count(*) tong, count(*)-count(skp_employee) trong_skp_employee, count(*)-count(amt_loan_limit) trong_loan_limit,
  count(*)-count(result) trong_result, sum((code_sa='XNA')::int) code_sa_XNA, sum((text_accepted_by='XNA')::int) accepted_by_XNA from leads''')

,tong,trong_skp_employee,trong_loan_limit,trong_result,code_sa_XNA,accepted_by_XNA
0,6130637,6130637,19372,20411,"1,434,972.00","205,789.00"


`skp_employee` trống ở 100% số dòng, nên không thể nối lead với nhân viên qua cột này.

Câu lệnh dưới đây xem `amt_loan_limit` trống ở loại sản phẩm nào.

In [ ]:
q('''select code_product_type, count(*) so_dong, count(*)-count(amt_loan_limit) trong_loan_limit from leads group by 1 order by 1''')

,code_product_type,so_dong,trong_loan_limit
0,CC,1245942,6790
1,CD,284167,6093
2,CL,4409471,1
3,REL,99219,0
4,TW,91838,6488


Trong 19.372 dòng trống hạn mức, 19.371 dòng thuộc ba loại CC (6.790), CD (6.093) và TW (6.488). Lead CL chỉ có 1 dòng trống, lead REL không có dòng nào.

Tiếp theo, câu lệnh dưới đây xem `code_sa` chứa những giá trị nào nhiều nhất.

In [ ]:
q('''select code_sa, count(*) so_dong from leads group by 1 order by 2 desc limit 6''')

,code_sa,so_dong
0,XNA,1434972
1,vn_api_fin,660435
2,BNPL_ONBOARDING_TECH_USER,26972
3,Koyal_User,15795
4,114272,5665
5,0032398,5600


Ngoài 1,43 triệu dòng XNA, `code_sa` còn chứa các tài khoản hệ thống chứ không phải người: "vn_api_fin" (660.435 dòng), "BNPL_ONBOARDING_TECH_USER", "Koyal_User". Nếu tính năng suất nhân viên theo `code_sa`, phải loại các tài khoản này.

**Kiểm tra 7.3. Hạn mức chào có giá trị nào không hợp lý không.** Hạn mức cao chưa chắc là sai, vì khách thu nhập cao có thể được chào hạn mức lớn. Để phân biệt, cần so hạn mức chào với số tiền vay thật trong bảng contract. Câu lệnh đầu dưới đây lấy số tiền vay lớn nhất từng có của mỗi loại sản phẩm trong bảng contract.

In [ ]:
q('''select product, max(credit_amount) vay_lon_nhat from contract where credit_amount>0 group by 1 order by 2 desc''')

,product,vay_lon_nhat
0,CL,"250,000,000.00"
1,CD,"100,000,000.00"
2,TW,"80,000,000.00"
3,BNPL,"50,000,000.00"
4,CW,"50,000,000.00"


Khoản vay tiền mặt (CL) lớn nhất là 250 triệu. Câu lệnh dưới đây chia hạn mức chào trong bảng leads thành các khoảng, rồi đếm số dòng mỗi khoảng.

In [ ]:
q('''select case when amt_loan_limit is null then '0. trống' when amt_loan_limit<1000000 then '1. dưới 1 triệu'
  when amt_loan_limit<=250000000 then '2. 1 triệu – 250 triệu' when amt_loan_limit<=300000000 then '3. trên 250 – 300 triệu'
  else '4. trên 300 triệu' end khoang, count(*) so_dong, min(amt_loan_limit) nho_nhat, max(amt_loan_limit) lon_nhat
from leads group by 1 order by 1''')

,khoang,so_dong,nho_nhat,lon_nhat
0,0. trống,19372,<NA>,<NA>
1,1. dưới 1 triệu,1,9,9
2,2. 1 triệu – 250 triệu,6111102,1000000,250000000
3,3. trên 250 – 300 triệu,153,250471800,281364800
4,4. trên 300 triệu,9,340000000,19000000000


- 153 dòng có hạn mức trên 250 đến 300 triệu, cao nhất khoảng 281 triệu. Mức này chỉ nhỉnh hơn khoản vay lớn nhất từng có (250 triệu), nên hợp lý với khách thu nhập cao. Giữ nguyên.
- Chỉ có 1 dòng dưới 1 triệu và 9 dòng trên 300 triệu. Câu lệnh dưới đây in 10 dòng này, kèm số hợp đồng ghi trong lead (`text_contract_number`) và số tiền vay thật của hồ sơ gắn với lead (`credit_amount` trong bảng contract).

In [ ]:
q('''select l.amt_loan_limit, l.text_contract_number, l.result, c.product, c.name_status, c.credit_amount
from leads l left join contract c using(skp_credit_case)
where l.amt_loan_limit<1000000 or l.amt_loan_limit>300000000 order by 1''')

,amt_loan_limit,text_contract_number,result,product,name_status,credit_amount
0,9,4605477754,8.00,CL,Cancelled,"9,000,000.00"
1,340000000,4603518103,1.00,CL,Cancelled,"40,000,000.00"
2,800000000,4506896798,2.00,CL,Cancelled,"80,000,000.00"
3,4600061825,4600061825,7.00,CL,Finished,"21,000,000.00"
4,4602835637,4602835637,2.00,CL,Cancelled,"69,200,000.00"
5,4604367781,4604367781,1.00,CL,Cancelled,"36,000,000.00"
6,4605019829,4605019829,7.00,CL,Active,"12,700,000.00"
7,4605022773,4605022773,3.00,CL,Cancelled,"100,000,000.00"
8,4605146527,4605146527,1.00,CL,Cancelled,"9,000,000.00"
9,19000000000,4605595267,7.00,None,None,NaN


10 dòng này là lỗi nhập liệu, không phải khách giàu:
- **6 dòng có hạn mức đúng bằng số hợp đồng** của chính lead đó, ví dụ hạn mức 4.600.061.825 và số hợp đồng 4600061825. Nghĩa là số hợp đồng đã bị nhập nhầm vào ô hạn mức.
- **Dòng hạn mức 9 đồng** gắn với hồ sơ vay 9 triệu. Nhiều khả năng hạn mức ghi theo đơn vị triệu đồng.
- **Dòng 340 triệu và dòng 800 triệu** gắn với hồ sơ vay 40 triệu và 80 triệu, tức hạn mức gấp khoảng 8–10 lần khoản vay. Nhiều khả năng bị nhập dư chữ số.
- **Dòng 19 tỷ** gấp 76 lần khoản vay lớn nhất từng có, và không tìm thấy hồ sơ nào gắn với lead này.

Cách xử lý: để trống hạn mức của 10 dòng này (dưới 1 triệu hoặc trên 300 triệu) khi tính trung bình. Không sửa thành giá trị đoán, vì không chắc giá trị đúng là bao nhiêu.

**Kiểm tra 7.4. Cột nào cho biết kết quả lead.** StructureTable ghi rằng phải tra `CODE_LEAD_STATUS` với `ID_SOURCE` của bảng cl_lead_fin_reason. Câu lệnh dưới đây đếm số dòng theo từng giá trị `code_lead_status`, tách theo cột `result` có trống hay không.

In [ ]:
q('''select code_lead_status, result is null as result_trong, count(*) so_dong from leads group by all order by all''')

,code_lead_status,result_trong,so_dong
0,1,True,20311
1,4,False,6110224
2,6,False,2
3,6,True,100


`code_lead_status` chỉ có 3 giá trị:
- giá trị 4 chiếm 6,11 triệu dòng, và **luôn có** kết quả ở cột `result`;
- giá trị 1 thì **không có** kết quả;
- giá trị 6 chỉ có khoảng 100 dòng, gần như không có kết quả.

Nếu tra `code_lead_status` theo bảng cl_lead_fin_reason như StructureTable hướng dẫn, gần như mọi lead sẽ thành "NO_CLUSTER" (mã 4). Điều này không hợp lý.

Câu lệnh dưới đây xem thêm các mã lead bị lặp: dòng tạo trước và dòng tạo sau có `code_lead_status` thế nào, và cách nhau bao nhiêu ngày.

In [ ]:
q('''with d as (select id_uuid from leads group by 1 having count(*)>1),
x as (select l.id_uuid, l.code_lead_status, l.result, l.dtime_created, row_number() over (partition by l.id_uuid order by l.dtime_created) rn from leads l semi join d using(id_uuid))
select a.code_lead_status status_dong_truoc, b.code_lead_status status_dong_sau, (b.result is null) dong_sau_khong_co_ket_qua, count(*) so_cap,
  median(date_diff('day',a.dtime_created,b.dtime_created)) trung_vi_cach_ngay
from x a join x b on a.id_uuid=b.id_uuid and a.rn=1 and b.rn=2 group by all order by so_cap desc''')

,status_dong_truoc,status_dong_sau,dong_sau_khong_co_ket_qua,so_cap,trung_vi_cach_ngay
0,4,4,False,204267,0.00
1,4,1,True,4723,4.00


Có 4.723 mã lead lúc đầu ở trạng thái 4 (có kết quả), khoảng 4 ngày sau xuất hiện lại ở trạng thái 1 (chưa có kết quả).

Từ hai kết quả trên, cách hiểu hợp lý là:
- `code_lead_status` là trạng thái của lead: 4 là đã đóng, 1 là đang mở;
- kết quả cuối cùng nằm ở cột `result`, và phải tra cột này với `ID_SOURCE`.

Vì trái với hướng dẫn trong StructureTable, nhóm nên hỏi lại Ban tổ chức cho chắc.

**Kiểm tra 7.5. Kết quả lead có khớp với hồ sơ gắn trong lead không.** Một số lead có sẵn mã hồ sơ ở cột `skp_credit_case`. Câu lệnh dưới đây nối các lead đó với bảng contract, rồi đếm theo kết quả lead: có bao nhiêu hồ sơ đã ký và bao nhiêu hồ sơ bị từ chối.

In [ ]:
q(f'''select l.result, count(*) lead_co_ho_so, sum((c.name_status in {SIGNED})::int) ho_so_da_ky, sum((c.name_status='Rejected')::int) ho_so_bi_tu_choi
from leads l join contract c using(skp_credit_case) group by 1 order by 1''')

,result,lead_co_ho_so,ho_so_da_ky,ho_so_bi_tu_choi
0,1.00,26468,"19,178.00",52.00
1,2.00,7104,"2,190.00",14.00
2,3.00,11445,"8,113.00",65.00
3,4.00,18,16.00,0.00
4,7.00,59325,"59,216.00",0.00
5,8.00,14739,50.00,"6,667.00"
6,11.00,72,70.00,0.00
7,NaN,631,511.00,3.00


- Có 26.468 lead ghi kết quả 1 (khách không quan tâm) nhưng lại gắn với một hồ sơ, và 19.178 hồ sơ trong số đó đã ký.
- Lead ghi kết quả 8 (bị từ chối) có 14.739 dòng gắn hồ sơ, nhưng chỉ 6.667 hồ sơ thực sự bị từ chối.

Như vậy `result` và `skp_credit_case` của lead không phải lúc nào cũng nói cùng một điều.

Câu lệnh dưới đây kiểm tra thêm: các mã hồ sơ được lead nhắc tới có tồn tại trong bảng contract hay không.

In [ ]:
q('''with l as (select distinct skp_credit_case from leads where skp_credit_case<>-1)
select count(*) ma_ho_so_trong_lead, sum((c.skp_credit_case is not null)::int) co_trong_contract from l left join contract c using(skp_credit_case)''')

,ma_ho_so_trong_lead,co_trong_contract
0,113837,"103,926.00"


Có 113.837 mã hồ sơ được lead nhắc tới, nhưng 9.911 mã không có trong bảng contract.

**Kiểm tra 7.6. Hồ sơ gắn sẵn trong lead có thỏa 3 điều kiện nối của đề không.** Đề quy định một lead dẫn tới một hợp đồng khi cùng khách, cùng sản phẩm, và lead được giao trước hoặc cùng lúc khách nộp hồ sơ. Câu lệnh dưới đây lấy các cặp lead–hồ sơ có sẵn trong bảng lead, nhóm theo cặp sản phẩm (sản phẩm của lead và sản phẩm của hồ sơ), rồi đếm:
- số cặp cùng khách;
- số cặp có lead được giao trước khi khách nộp hồ sơ;
- số cặp có lead và hồ sơ cùng điểm bán.

In [ ]:
q('''select l.code_product_type lead_product, c.product contract_product, count(*) so_cap,
  sum((l.skp_client=c.skp_client)::int) cung_khach, sum((l.date_assigned<=c.date_apply)::int) giao_lead_truoc_khi_nop,
  sum((l.skp_salesroom=c.skp_salesroom)::int) cung_diem_ban
from leads l join contract c using(skp_credit_case) group by all order by 3 desc''')

,lead_product,contract_product,so_cap,cung_khach,giao_lead_truoc_khi_nop,cung_diem_ban
0,CL,CL,62163,"62,104.00","21,015.00","3,873.00"
1,CC,CC,56132,"56,124.00","47,345.00","17,645.00"
2,REL,CC,821,821.00,807.00,0.00
3,TW,TW,373,373.00,352.00,210.00
4,CL,CW,305,303.00,0.00,0.00
5,CC,CL,4,4.00,0.00,0.00
6,CL,CC,4,4.00,0.00,0.00


- Với lead vay tiền mặt (CL) gắn hồ sơ CL, chỉ 21.015 trên 62.163 cặp có lead được giao trước khi khách nộp hồ sơ. Khoảng 2/3 còn lại có hồ sơ nộp **trước** khi lead được giao, tức hồ sơ được gắn vào lead sau khi khách đã nộp. Các cặp này vi phạm điều kiện thứ 3 của đề và không được tính là lead dẫn tới hợp đồng.
- Có 305 cặp lead CL gắn hồ sơ CW, và 821 cặp lead REL gắn hồ sơ CC, tức sản phẩm không khớp.

Câu lệnh dưới đây xem lead theo từng loại sản phẩm: lead đến từ nguồn nào, có bao nhiêu lead gắn sẵn hồ sơ, và bao nhiêu lead chốt được (`result = 7`).

In [ ]:
q('''select code_product_type, string_agg(distinct id_lead_source, ', ') nguon, count(*) so_dong,
  sum((skp_credit_case<>-1)::int) co_ho_so, sum((result=7)::int) chot_duoc from leads group by 1 order by 3 desc''')

,code_product_type,nguon,so_dong,co_ho_so,chot_duoc
0,CL,"INSTANT_ONE_MOB, WDE, GMA_BIG_TICKET, CRM, INSTANT_TWO_RTDM, INSTANT_ONE_ALDI, GMA, SOB",4409471,"67,924.00","25,385.00"
1,CC,"INSTANT_ONE_ALDI, GMA, INSTANT_ONE_MOB, WDE",1245942,"61,242.00","39,212.00"
2,CD,SOB,284167,0.00,0.00
3,REL,INSTANT_ONE_ALDI,99219,821.00,138.00
4,TW,"GMA_SCORING, GMA, SOB",91838,394.00,178.00


Lead CD (284.167 dòng, chỉ đến từ nguồn SOB) không gắn với hồ sơ nào và không có lead nào chốt được. Với phần bán chéo CD, nhóm chỉ có thể dựa vào việc tự nối theo 3 điều kiện ở Kiểm tra 7.7.

**Kiểm tra 7.7. Tự nối lead với contract đúng theo 3 điều kiện của đề.** Câu lệnh dưới đây không dùng mã hồ sơ có sẵn trong lead. Nó tự tìm, cho mỗi lead, mọi hồ sơ cùng khách, cùng sản phẩm (CL, CC, TW, CD), và nộp từ lúc lead được giao trở đi. Sau đó tổng hợp:
- số cặp lead–hồ sơ tìm được, số lead và số hồ sơ có cặp;
- số hồ sơ nối với từ 2 lead trở lên, và hồ sơ nối với nhiều lead nhất;
- khoảng cách từ lúc giao lead tới lúc nộp hồ sơ: trung vị và mức 75%;
- số lead DEAL (chốt được) tìm thấy hồ sơ, so với tổng số lead DEAL;
- số lead "khách không quan tâm" vẫn tìm thấy hồ sơ.

In [ ]:
q('''with l as (select row_number() over () lid, skp_client, code_product_type p, date_assigned, result from leads where code_product_type in ('CL','CC','TW','CD')),
c as (select skp_credit_case, skp_client::BIGINT skp_client, product p, date_apply from contract where product in ('CL','CC','TW','CD') and skp_client is not null),
m as (select l.lid, l.result, c.skp_credit_case, date_diff('day', l.date_assigned, c.date_apply) cach_ngay
      from l join c on l.skp_client=c.skp_client and l.p=c.p and c.date_apply>=l.date_assigned),
pc as (select skp_credit_case, count(*) k from m group by 1)
select (select count(*) from m) so_cap, (select count(distinct lid) from m) so_lead_noi_duoc, (select count(*) from pc) so_hd_noi_duoc,
  (select sum((k>1)::int) from pc) hd_noi_voi_nhieu_lead, (select max(k) from pc) nhieu_lead_nhat_cho_1_hd,
  (select quantile_cont(cach_ngay,0.5) from m) trung_vi_ngay, (select quantile_cont(cach_ngay,0.75) from m) p75_ngay,
  (select count(distinct lid) from m where result=7) lead_deal_noi_duoc, (select count(*) from l where result=7) tong_lead_deal,
  (select count(distinct lid) from m where result=1) lead_khong_quan_tam_van_noi_duoc''')

,so_cap,so_lead_noi_duoc,so_hd_noi_duoc,hd_noi_voi_nhieu_lead,nhieu_lead_nhat_cho_1_hd,trung_vi_ngay,p75_ngay,lead_deal_noi_duoc,tong_lead_deal,lead_khong_quan_tam_van_noi_duoc
0,339529,295588,260674,"54,703.00",319,11.00,86.00,38717,64775,200090


Áp đúng 3 điều kiện của đề cho ra 339.529 cặp. Có ba vấn đề:

- **Một hợp đồng nối với nhiều lead.** 54.703 hợp đồng nối được với từ 2 lead trở lên, nhiều nhất là 319 lead cho một hợp đồng. Nếu cộng doanh số theo lead, một hợp đồng sẽ bị tính cho nhiều shop. Cần quy tắc chọn một lead duy nhất cho mỗi hợp đồng, ví dụ lead được giao gần nhất trước ngày nộp.
- **Không có giới hạn thời gian.** Trung vị khoảng cách là 11 ngày, nhưng 25% số cặp cách nhau trên 86 ngày. Một lead giao từ tháng 1 vẫn "dẫn tới" hợp đồng nộp tháng 6. Nhóm cần chốt một khoảng thời gian tối đa (ví dụ 30 hoặc 90 ngày) và ghi rõ trong slide.
- **Kết quả lead và việc nối được không khớp.**
  - Chỉ 38.717 trên 64.775 lead DEAL tìm thấy hợp đồng theo quy tắc của đề.
  - Ngược lại, 200.090 lead ghi "khách không quan tâm" vẫn nối được với một hợp đồng.

**Kiểm tra 7.8. Lead gắn với shop có trong bảng salesroom không.** Câu lệnh đầu đếm số mã điểm bán xuất hiện trong lead và số mã có trong bảng salesroom. Câu lệnh sau đếm số dòng lead và số lead sau khi nhân `sample_weight`, tách theo shop có trong bảng hay không.

In [ ]:
q('''with x as (select distinct skp_salesroom from leads)
select count(*) so_ma_diem_ban_trong_lead, sum((s.SKP_SALESROOM is not null)::int) co_trong_bang_salesroom from x left join salesroom s on x.skp_salesroom=s.SKP_SALESROOM::BIGINT''')

,so_ma_diem_ban_trong_lead,co_trong_bang_salesroom
0,27380,"10,796.00"


In [ ]:
q('''select (s.SKP_SALESROOM is not null) shop_trong_bang, count(*) so_dong, round(sum(l.sample_weight)) so_lead_sau_trong_so
from leads l left join salesroom s on l.skp_salesroom=s.SKP_SALESROOM::BIGINT group by 1''')

,shop_trong_bang,so_dong,so_lead_sau_trong_so
0,False,5745364,"42,177,859.00"
1,True,385273,"1,276,986.00"


Lead gắn với 27.380 mã điểm bán, nhưng chỉ 10.796 mã có trong bảng salesroom. Tính theo số lead đã nhân `sample_weight`, chỉ khoảng 1,28 triệu trên 43,5 triệu lead (khoảng 3%) gắn với shop có trong bảng.

Phần cross-sell mà nhóm tính được cho từng shop vì vậy chỉ là một phần nhỏ của toàn bộ lead. Cần ghi rõ giới hạn này.

**Kiểm tra 7.9. `code_pos` và `skp_salesroom` có đi thành cặp cố định không.** Nếu mỗi `code_pos` luôn ứng với đúng một `skp_salesroom`, thì số mã của hai cột và số cặp phải bằng nhau.

In [ ]:
q('''select count(distinct code_pos) so_code_pos, count(distinct skp_salesroom) so_skp_salesroom, count(distinct (code_pos, skp_salesroom)) so_cap from leads''')

,so_code_pos,so_skp_salesroom,so_cap
0,27380,27380,27380


Ba con số bằng nhau (27.380), nên hai cột đi thành cặp 1–1. Dùng cột nào để nhận diện điểm bán cũng được.

**Kiểm tra 7.10. Có lead nào được giao trước khi được tạo không.** Câu lệnh dưới đây đếm số lead có `date_assigned` sớm hơn `dtime_created`.

In [ ]:
q('''select sum((date_assigned<dtime_created)::int) giao_truoc_khi_tao from leads''')

,giao_truoc_khi_tao
0,20.00


Có 20 lead như vậy. Số lượng rất nhỏ, có thể loại.

## Phần 8. Quy mô của từng vấn đề

Feedback Round 2 nhận xét nhóm đã tìm ra lỗi dữ liệu nhưng chưa nói lỗi lớn đến đâu. Phần này đo quy mô: mỗi vấn đề ảnh hưởng bao nhiêu dòng, trên tổng bao nhiêu dòng, tức bao nhiêu phần trăm của bảng.

**Kiểm tra 8.1. Quy mô các vấn đề đã tìm thấy.** Câu lệnh dưới đây tính lại từng con số từ dữ liệu gốc. Mỗi dòng của kết quả gồm: tên vấn đề, số dòng bị ảnh hưởng (`so_dong_loi`), tổng số dòng dùng để so (`tren_tong`), và tỷ lệ phần trăm.

In [ ]:
TINH_CU_LIST = TINH_CU
q(f'''
select 'Hợp đồng đã ký có ngày ký 01/2025–02/2025 hoặc từ 08/2026 (DPD lệch)' van_de,
  (select count(*) from contract where name_status in {SIGNED} and (date_sign<'2025-03-01' or date_sign>='2026-08-01')) so_dong_loi, (select count(*) from contract where name_status in {SIGNED}) tren_tong
union all select 'Hồ sơ TW ở điểm bán không có trong bảng salesroom',
  (select count(*) from contract c left join salesroom s on c.skp_salesroom=s.SKP_SALESROOM::BIGINT where c.product='TW' and s.SKP_SALESROOM is null), (select count(*) from contract where product='TW')
union all select 'Lead (sau trọng số) gắn với điểm bán không có trong bảng salesroom',
  (select round(sum(l.sample_weight)) from leads l left join salesroom s on l.skp_salesroom=s.SKP_SALESROOM::BIGINT where s.SKP_SALESROOM is null), (select round(sum(sample_weight)) from leads)
union all select 'Hợp đồng CC, BNPL đã ký có credit_amount = 0',
  (select count(*) from contract where name_status in {SIGNED} and product in ('CC','BNPL') and credit_amount=0), (select count(*) from contract where name_status in {SIGNED} and product in ('CC','BNPL'))
union all select 'Hồ sơ có date_sign = 01/01/3000 (chưa đi tới bước ký)', (select count(*) from contract where year(date_sign)=3000), (select count(*) from contract)
union all select 'Phân công có dtime_valid_to = 01/01/3000 (chưa kết thúc)', (select count(*) from e2s where year(dtime_valid_to)=3000), (select count(*) from e2s)
union all select 'Dòng trùng hoàn toàn trong emp_2_salesroom', (select count(*) from e2s)-(select count(*) from (select distinct * from e2s)), (select count(*) from e2s)
union all select 'Dòng dư trong contract_product (hồ sơ có nhiều dòng)', (select count(*) from cp)-(select count(distinct skp_credit_case) from cp), (select count(*) from cp)
union all select 'Dòng lead có id_uuid bị lặp', (select count(*) from leads)-(select count(distinct id_uuid) from leads), (select count(*) from leads)
union all select 'Shop ghi tên tỉnh cũ', (select count(*) from salesroom where SALESROOM_PROVINCE in {TINH_CU_LIST}), (select count(*) from salesroom)
union all select 'Shop ghi địa chỉ theo địa giới cũ (còn tên quận)', (select count(*) from salesroom where SALESROOM_DISTRICT<>'.'), (select count(*) from salesroom)
union all select 'Dòng contract_product trống goods_group', (select count(*) from cp where goods_group is null), (select count(*) from cp)
union all select 'Lead có code_sa là XNA hoặc tài khoản hệ thống', (select count(*) from leads where code_sa='XNA' or not regexp_matches(code_sa,'^[0-9]+$')), (select count(*) from leads)
union all select 'Lead có hạn mức trống hoặc nhập sai (dưới 1 triệu, trên 300 triệu)', (select count(*) from leads where amt_loan_limit is null or amt_loan_limit<1000000 or amt_loan_limit>300000000), (select count(*) from leads)
''').assign(phan_tram=lambda d: (100*d.so_dong_loi/d.tren_tong).round(2))

,van_de,so_dong_loi,tren_tong,phan_tram
0,Hợp đồng đã ký có ngày ký 01/2025–02/2025 hoặc từ 08/2026 (DPD lệch),"535,347.00","2,715,985.00",19.71
1,Hồ sơ TW ở điểm bán không có trong bảng salesroom,"255,858.00","329,335.00",77.69
2,Lead (sau trọng số) gắn với điểm bán không có trong bảng salesroom,"42,177,859.00","43,454,845.00",97.06
3,"Hợp đồng CC, BNPL đã ký có credit_amount = 0","894,400.00","1,136,753.00",78.68
4,Hồ sơ có date_sign = 01/01/3000 (chưa đi tới bước ký),"1,048,871.00","3,775,371.00",27.78
5,Phân công có dtime_valid_to = 01/01/3000 (chưa kết thúc),"233,807.00","275,856.00",84.76
6,Dòng trùng hoàn toàn trong emp_2_salesroom,912.00,"275,856.00",0.33
7,Dòng dư trong contract_product (hồ sơ có nhiều dòng),"4,961.00","784,902.00",0.63
8,Dòng lead có id_uuid bị lặp,"208,991.00","6,130,637.00",3.41
9,Shop ghi tên tỉnh cũ,112.00,"14,231.00",0.79


Bảng trên chia các vấn đề thành hai loại rõ rệt:

- **Vấn đề lớn, ảnh hưởng trực tiếp tới kết luận:**
  - khoảng 78% hồ sơ TW và khoảng 97% lead nằm ngoài các shop trong bảng;
  - khoảng 79% hợp đồng CC/BNPL đã ký có `credit_amount` bằng 0;
  - khoảng 36% dòng lead có `code_sa` là XNA hoặc tài khoản hệ thống, không phải mã nhân viên;
  - khoảng 20% hợp đồng đã ký rơi vào các tháng có DPD bị lệch.

  Những vấn đề này nếu không xử lý sẽ làm sai rủi ro, sai tổng tiền cho vay, sai năng suất nhân viên theo lead, và phải được nêu trong slide.
- **Vấn đề nhỏ, dưới 5% số dòng:** dòng trùng, tên tỉnh cũ, ô trống `goods_group`, hạn mức nhập sai. Xử lý đơn giản, chỉ cần ghi trong file đính kèm.

Riêng giá trị 01/01/3000 chiếm nhiều dòng (khoảng 28% hồ sơ và 85% phân công), nhưng đây là giá trị quy ước, không phải lỗi. Chỉ cần xử lý đúng khi tính theo ngày.

## Phần 9. Thử cách xử lý và so sánh trước – sau

Feedback Round 2 yêu cầu giải thích vì sao chọn mỗi cách xử lý, và cho thấy kết quả trước và sau khi xử lý. Phần này làm đúng việc đó cho từng vấn đề:
1. nêu các cách xử lý có thể;
2. chọn một cách và giải thích vì sao;
3. chạy thử, rồi so con số trước và sau.

Các bảng sau khi xử lý ở đây chỉ tạo tạm trong bộ nhớ để kiểm tra, chưa ghi ra file. Người làm bước làm sạch dùng lại đúng các câu lệnh này.

### 9.1. Đổi tên tỉnh cũ về tỉnh mới

Theo Nghị quyết 202/2025, mỗi tỉnh cũ được gộp vào đúng một tỉnh mới, nên chỉ có một cách đổi hợp lý. Bảng đổi dưới đây ghi tỉnh cũ và tỉnh mới tương ứng. Câu lệnh so số tên tỉnh khác nhau trước và sau khi đổi. Theo địa giới mới, cả nước có 34 tỉnh, thành.

In [ ]:
DOI_TINH = {'Bình Định':'Gia Lai','Bạc Liêu':'Cà Mau','Bắc Giang':'Bắc Ninh','Bắc Kạn':'Thái Nguyên','Bến Tre':'Vĩnh Long',
  'Hà Giang':'Tuyên Quang','Hà Nam':'Ninh Bình','Hải Dương':'Hải Phòng','Kiên Giang':'An Giang','Kon Tum':'Quảng Ngãi',
  'Nam Định':'Ninh Bình','Ninh Thuận':'Khánh Hòa','Phú Yên':'Đắk Lắk','Quảng Bình':'Quảng Trị','Quảng Nam':'Đà Nẵng',
  'Tiền Giang':'Đồng Tháp','Trà Vinh':'Vĩnh Long','Vĩnh Phúc':'Phú Thọ','Yên Bái':'Lào Cai','Đắk Nông':'Lâm Đồng'}
doi = pd.DataFrame(list(DOI_TINH.items()), columns=['tinh_cu','tinh_moi']); con.register('doi', doi)
q('''select (select count(distinct SALESROOM_PROVINCE) from salesroom) so_tinh_truoc,
  (select count(distinct coalesce(d.tinh_moi, s.SALESROOM_PROVINCE)) from salesroom s left join doi d on s.SALESROOM_PROVINCE=d.tinh_cu) so_tinh_sau,
  (select count(distinct SELLER_PROVINCE) from seller) so_tinh_seller_truoc,
  (select count(distinct coalesce(d.tinh_moi, x.SELLER_PROVINCE)) from seller x left join doi d on x.SELLER_PROVINCE=d.tinh_cu) so_tinh_seller_sau''')

,so_tinh_truoc,so_tinh_sau,so_tinh_seller_truoc,so_tinh_seller_sau
0,54,34,54,34


Sau khi đổi, cả bảng salesroom và bảng seller về đúng 34 tỉnh, thành, khớp với địa giới mới.

### 9.2. Chia peer group theo tỉnh mới

Peer group là nhóm các shop giống nhau, dùng để so shop với shop cho công bằng. Shop cùng nhóm phải cùng cỡ shop, cùng ngành hàng, cùng kiểu Aldi hoặc Non-Aldi, và cùng khu vực.

Nhóm chọn khu vực là **tỉnh mới** (34 tỉnh), vì dễ hiểu và dễ giải thích với ban quản lý. Cách này có hai đánh đổi. Hai câu lệnh dưới đây cho thấy hai đánh đổi đó.

**Đánh đổi 1. Một số nhóm có quá ít shop.** Câu lệnh dưới đây chia các shop đang Active (trừ shop online) thành peer group theo tỉnh mới, rồi đếm số nhóm, số shop trung vị mỗi nhóm, và số nhóm dưới 20 shop.

In [ ]:
q('''with t as (select SALESROOM_CATEGORY cat, SALESROOM_PRODUCT prod, SALESROOM_MODEL model, coalesce(d.tinh_moi, s.SALESROOM_PROVINCE) tinh
  from salesroom s left join doi d on s.SALESROOM_PROVINCE=d.tinh_cu
  where SALESROOM_STATUS='Active' and SALESROOM_CATEGORY<>'Virtual'),
g as (select count(*) n from t group by cat, prod, model, tinh)
select count(*) so_peer_group, median(n) shop_trung_vi_moi_nhom, sum((n<20)::int) nhom_duoi_20_shop,
  sum(case when n<20 then n end) shop_trong_nhom_nho, sum(n) tong_shop from g''')

,so_peer_group,shop_trung_vi_moi_nhom,nhom_duoi_20_shop,shop_trong_nhom_nho,tong_shop
0,328,15.00,187.00,"1,350.00","11,767.00"


Có 187 nhóm dưới 20 shop, chứa 1.350 shop (11,5%). Với các nhóm này, mức chuẩn "1/4 số shop đứng đầu" chỉ dựa trên vài shop, nên kết quả của những shop đó phải gắn nhãn độ tin cậy thấp.

**Đánh đổi 2. Một tỉnh mới có thể gộp nhiều khu vực khác nhau.** Câu lệnh dưới đây lấy các shop CD cỡ Small, Non-Aldi, đang Active ở TP.HCM mới, rồi đếm theo vùng kinh doanh. Như Kiểm tra 3.4 đã cho thấy, mỗi vùng ứng với một tỉnh cũ.

In [ ]:
q('''select coalesce(d.tinh_moi, s.SALESROOM_PROVINCE) tinh_moi, SALESROOM_AREA vung, count(*) so_shop
from salesroom s left join doi d on s.SALESROOM_PROVINCE=d.tinh_cu
where SALESROOM_STATUS='Active' and SALESROOM_CATEGORY='Small' and SALESROOM_PRODUCT='CD' and SALESROOM_MODEL='Non-Aldi'
  and coalesce(d.tinh_moi, s.SALESROOM_PROVINCE)='TP.HCM'
group by all order by 3 desc''')

,tinh_moi,vung,so_shop
0,TP.HCM,Sai Gon,215
1,TP.HCM,South East,120
2,TP.HCM,Eastern South,35


Khoảng 370 shop này nằm chung một peer group "Small – CD – Non-Aldi – TP.HCM", dù 215 shop ở TP.HCM cũ, 120 shop ở Bình Dương cũ và 35 shop ở Bà Rịa – Vũng Tàu cũ. Khi trình bày cần nói rõ điều này.

### 9.3. Bỏ dòng trùng trong emp_2_salesroom

Dòng trùng hoàn toàn không mang thêm thông tin, nên chỉ có một cách hợp lý là giữ một dòng. Câu lệnh dưới đây so số dòng trước và sau khi bỏ trùng, và kiểm tra sau khi bỏ thì không còn dòng nào trùng.

In [ ]:
q('''select (select count(*) from e2s) truoc, (select count(*) from (select distinct * from e2s)) sau,
  (select count(*) from (select distinct * from e2s)) - (select count(*) from (select distinct * from (select distinct * from e2s))) con_trung_sau_xu_ly''')

,truoc,sau,con_trung_sau_xu_ly
0,275856,274944,0


Số dòng giảm từ 275.856 xuống 274.944, và sau xử lý không còn dòng trùng.

### 9.4. Xử lý giá trị 01/01/3000

- Ở `date_sign`: 01/01/3000 nghĩa là hồ sơ chưa đi tới bước ký. Cách xử lý là đổi thành ô trống, vì hồ sơ đó không có ngày ký thật.
- Ở `dtime_valid_to`: 01/01/3000 nghĩa là phân công chưa kết thúc. Cách xử lý là thay bằng ngày cuối của dữ liệu (17/10/2026), vì đến ngày đó nhân viên vẫn đang làm.

Câu lệnh dưới đây so trước và sau khi xử lý:
- số hồ sơ rơi vào "năm 3000" khi đếm theo năm ký;
- thời gian phân công trung bình, tính bằng ngày.

In [ ]:
q('''select
  (select count(*) from contract where year(date_sign)=3000) ho_so_nam_3000_truoc,
  (select count(*) from (select case when year(date_sign)=3000 then null else date_sign end ds from contract) where year(ds)=3000) ho_so_nam_3000_sau,
  (select round(avg(date_diff('day',dtime_valid_from,dtime_valid_to))) from e2s) so_ngay_phan_cong_tb_truoc,
  (select round(avg(date_diff('day',dtime_valid_from, case when year(dtime_valid_to)=3000 then timestamp '2026-10-17 23:59:59' else dtime_valid_to end))) from e2s) so_ngay_phan_cong_tb_sau''')

,ho_so_nam_3000_truoc,ho_so_nam_3000_sau,so_ngay_phan_cong_tb_truoc,so_ngay_phan_cong_tb_sau
0,1048871,0,"301,530.00",255.00


Sau khi xử lý không còn hồ sơ nào nằm ở "năm 3000". Thời gian phân công trung bình giảm từ hàng trăm nghìn ngày xuống vài trăm ngày, đúng với khoảng thời gian thật của dữ liệu.

### 9.5. Đếm số SA của shop

Có hai nguồn để biết shop có những nhân viên SA nào: bảng emp_2_salesroom (phân công) và cột `skp_emp_pre_process` của bảng contract (người làm hồ sơ). Hai nguồn này cho kết quả khác nhau. Câu lệnh dưới đây lấy ví dụ nhân viên 19231185: với mỗi shop người này đang được giao trong bảng phân công, đếm số hồ sơ CD/TW (mua hàng trả góp tại shop) người này thực sự làm tại shop đó trong quý 3/2026 (07–09/2026).

In [ ]:
q('''select e.skp_salesroom, count(c.skp_credit_case) so_ho_so_lam_trong_q3
from (select distinct skp_salesroom from e2s where skp_employee=19231185 and code_status='a') e
left join contract c on c.skp_salesroom=e.skp_salesroom and c.skp_emp_pre_process=19231185 and c.product in ('CD','TW') and c.date_apply>='2026-07-01'
group by 1 order by 2 desc''')

,skp_salesroom,so_ho_so_lam_trong_q3
0,123081950,130
1,106809022,0
2,121429165,0
3,109296212,0
4,121501209,0
5,126375952,0
6,99603619,0
7,121069003,0
8,124312717,0
9,99546340,0


Nhân viên này đang được giao ở 11 shop, nhưng trong quý 3/2026 chỉ làm hồ sơ ở 1 shop, với 130 hồ sơ. Vì vậy cần chọn một trong hai cách đếm:
- **Cách 1. Đếm nhân viên có phân công đang mở tại shop.**
  - Ưu điểm: có sẵn trong bảng phân công, kể cả nhân viên chưa làm hồ sơ nào.
  - Nhược điểm: một người được giao nhiều shop sẽ được tính là SA của tất cả các shop đó, kể cả shop người đó không làm hồ sơ nào.
- **Cách 2. Đếm nhân viên khác nhau thực sự làm hồ sơ tại shop trong tháng.**
  - Ưu điểm: phản ánh người thực sự làm việc tại shop. Người làm ở 2–3 shop vẫn được tính là SA của từng shop mình có làm hồ sơ.
  - Nhược điểm: bỏ sót nhân viên có mặt nhưng cả tháng không làm được hồ sơ nào, nên shop rất ế có thể bị đếm thiếu.

Câu lệnh dưới đây tính cả hai cách cho tháng 09/2026, chỉ với các shop có hồ sơ CD/TW trong tháng. Cách 1 đếm nhân viên có phân công còn hiệu lực vào ngày 15/09/2026. Kết quả so trung vị và mức 90% của số SA mỗi shop, và số hồ sơ mỗi SA theo từng cách.

In [ ]:
q('''with ho_so as (select c.skp_salesroom, count(*) n_ho_so, count(distinct c.skp_emp_pre_process) sa_cach2
  from contract c join salesroom s on c.skp_salesroom=s.SKP_SALESROOM::BIGINT
  where c.product in ('CD','TW') and c.date_apply>='2026-09-01' and c.date_apply<'2026-10-01' group by 1),
pc as (select skp_salesroom, count(distinct skp_employee) sa_cach1 from (select distinct * from e2s)
  where timestamp '2026-09-15' between dtime_valid_from and dtime_valid_to group by 1),
x as (select h.*, coalesce(p.sa_cach1,0) sa_cach1 from ho_so h left join pc p using(skp_salesroom))
select count(*) so_shop,
  quantile_cont(sa_cach1,0.5) sa_cach1_trung_vi, quantile_cont(sa_cach1,0.9) sa_cach1_p90,
  quantile_cont(sa_cach2,0.5) sa_cach2_trung_vi, quantile_cont(sa_cach2,0.9) sa_cach2_p90,
  round(sum(n_ho_so)/sum(nullif(sa_cach1,0)),2) ho_so_moi_sa_cach1, round(sum(n_ho_so)/sum(sa_cach2),2) ho_so_moi_sa_cach2
from x''')

,so_shop,sa_cach1_trung_vi,sa_cach1_p90,sa_cach2_trung_vi,sa_cach2_p90,ho_so_moi_sa_cach1,ho_so_moi_sa_cach2
0,6375,20.00,27.00,1.00,3.00,0.45,4.83


Theo cách 1, một shop điển hình có 20 "SA", và mỗi SA chỉ làm chưa tới 1 hồ sơ mỗi tháng. Nếu dùng con số này để tính năng suất, shop nào cũng có năng suất rất thấp, khó so sánh giữa các shop. Cách 2 cho trung vị 1 SA mỗi shop, mỗi SA làm khoảng 5 hồ sơ mỗi tháng, và vẫn tính đúng trường hợp một người làm ở 2–3 shop: người đó được tính là SA của từng shop mình có làm hồ sơ.

Nhóm chọn **cách 2**, và ghi rõ nhược điểm: có thể đếm thiếu ở shop rất ế.

### 9.6. Chọn hợp đồng để tính tỷ lệ trễ hạn

Có hai cách:
- **Cách 1. Dùng mọi hợp đồng đã ký.**
  - Ưu điểm: nhiều dữ liệu.
  - Nhược điểm: hợp đồng chưa đến kỳ trả bị tính là trả đúng hạn. Hợp đồng đầu năm 2025 có tỷ lệ trễ thấp bất thường.
- **Cách 2. Chỉ dùng hợp đồng ký từ 03/2025 đến hết 07/2026** khi tính trễ kỳ 1.
  - Ưu điểm: mọi hợp đồng đều đã qua kỳ trả đầu tiên, và tránh được giai đoạn ghi nhận thấp bất thường.
  - Nhược điểm: bỏ khoảng 20% hợp đồng.

Câu lệnh dưới đây so tỷ lệ trễ kỳ 1 theo tháng ký giữa hai cách: tháng thấp nhất, tháng cao nhất, và tỷ lệ chung.

In [ ]:
q(f'''with m as (select date_trunc('month',date_sign) thang, avg(dpd_1pd)*100 pct from contract where name_status in {SIGNED} group by 1)
select 'Cách 1: mọi hợp đồng' cach, round(min(pct),2) thang_thap_nhat, round(max(pct),2) thang_cao_nhat,
  (select round(avg(dpd_1pd)*100,2) from contract where name_status in {SIGNED}) ty_le_chung from m
union all
select 'Cách 2: ký 03/2025–07/2026', round(min(pct),2), round(max(pct),2),
  (select round(avg(dpd_1pd)*100,2) from contract where name_status in {SIGNED} and date_sign>='2025-03-01' and date_sign<'2026-08-01')
from m where thang>='2025-03-01' and thang<'2026-08-01' ''')

,cach,thang_thap_nhat,thang_cao_nhat,ty_le_chung
0,Cách 1: mọi hợp đồng,0.00,8.79,6.71
1,Cách 2: ký 03/2025–07/2026,6.76,8.79,7.46


Với cách 1, tỷ lệ trễ kỳ 1 theo tháng dao động từ 0% đến gần 9%. Mức thấp là do dữ liệu, không phải do khách tốt hơn. Với cách 2, các tháng chỉ dao động trong khoảng 6,8–8,8%, và tỷ lệ chung cao hơn. Nhóm chọn **cách 2**.

### 9.7. Giữ một dòng cho mỗi hồ sơ trong contract_product

Có hai cách:
- **Cách 1. Giữ nguyên nhiều dòng.**
  - Ưu điểm: không mất thông tin.
  - Nhược điểm: nối với contract thì số hợp đồng bị đếm dư.
- **Cách 2. Giữ một dòng cho mỗi hồ sơ.** Ưu tiên dòng có `goods_group`, sau đó ưu tiên dòng có loại hàng cụ thể hơn, tức không bắt đầu bằng "Other".
  - Ưu điểm: số hợp đồng đúng, và giữ được thông tin chi tiết nhất.
  - Nhược điểm: bỏ đi một số dòng thông tin phụ.

Nhóm chọn **cách 2**. Câu lệnh dưới đây so số hợp đồng CD/TW khi nối contract với contract_product, trước và sau khi xử lý, với số hợp đồng CD/TW thật trong bảng contract.

In [ ]:
q('''with cp1 as (select * from cp qualify row_number() over (partition by skp_credit_case
       order by (goods_group is null), (name_goods_type ilike 'other%'), name_goods_type) = 1)
select (select count(*) from contract where product in ('CD','TW')) hop_dong_cd_tw_that,
  (select count(*) from contract c join cp using(skp_credit_case) where c.product in ('CD','TW')) sau_khi_noi_truoc_xu_ly,
  (select count(*) from contract c join cp1 using(skp_credit_case) where c.product in ('CD','TW')) sau_khi_noi_sau_xu_ly''')

,hop_dong_cd_tw_that,sau_khi_noi_truoc_xu_ly,sau_khi_noi_sau_xu_ly
0,779943,784902,779941


Trước khi xử lý, phép nối cho ra nhiều hơn số hợp đồng thật. Sau khi xử lý, số dòng sau nối khớp với số hợp đồng thật. Chỉ còn lệch 2 hồ sơ CD không có dòng nào trong contract_product.

### 9.8. Thống nhất tên nhóm hàng và điền ô trống

Cách xử lý:
- gộp các tên chỉ khác nhau ở chữ hoa, chữ thường: "FITNESS" vào "Fitness", "CONSTRUCTION" vào "Construction";
- gộp "FURNITURE & MATTRESSES" vào "Furniture", và "OTHER" vào "Others";
- ô trống thì điền theo nhóm hàng lớn: xe máy mới vào "Two Wheelers", xe máy cũ vào "Used TW", fitness vào "Fitness", còn lại vào "Others".

Câu lệnh dưới đây so số tên nhóm hàng khác nhau và số ô trống, trước và sau khi xử lý.

In [ ]:
q('''with x as (select *, case
   when goods_group is null and name_goods_category='Two weelers' then 'Two Wheelers'
   when goods_group is null and name_goods_category='USED MOTORBIKE' then 'Used TW'
   when goods_group is null and name_goods_category='FITNESS' then 'Fitness'
   when goods_group is null then 'Others'
   when upper(goods_group) in ('FURNITURE & MATTRESSES','FURNITURE') then 'Furniture'
   when upper(goods_group) in ('OTHER','OTHERS') then 'Others'
   when upper(goods_group)='FITNESS' then 'Fitness'
   when upper(goods_group)='CONSTRUCTION' then 'Construction'
   else goods_group end goods_group_moi from cp)
select count(distinct goods_group) so_ten_truoc, count(*)-count(goods_group) o_trong_truoc,
  count(distinct goods_group_moi) so_ten_sau, count(*)-count(goods_group_moi) o_trong_sau from x''')

,so_ten_truoc,o_trong_truoc,so_ten_sau,o_trong_sau
0,20,12476,16,0


Số tên nhóm hàng giảm từ 20 xuống 16 vì 4 cặp tên trùng nghĩa đã được gộp, và không còn ô trống.

### 9.9. Nối lead với hợp đồng: mỗi hợp đồng một lead, có giới hạn thời gian

Với một hợp đồng nối được với nhiều lead (Kiểm tra 7.7), có hai cách ghi công:
- **Cách 1. Chia đều hợp đồng cho mọi lead nối được.**
  - Ưu điểm: không phải chọn lead nào.
  - Nhược điểm: khó giải thích với ban quản lý. Lead giao từ nhiều tháng trước vẫn được ghi công ngang lead mới.
- **Cách 2. Ghi công cho lead được giao gần nhất trước ngày nộp hồ sơ.**
  - Ưu điểm: đơn giản, dễ giải thích: "lần chào gần nhất trước khi khách vay".
  - Nhược điểm: bỏ qua các lần chào trước đó.

Ngoài ra phải chọn khoảng thời gian tối đa từ lúc giao lead tới lúc nộp hồ sơ:
- **30 ngày:** chắc chắn hơn rằng lead dẫn tới hợp đồng, nhưng bỏ sót khách cân nhắc lâu.
- **90 ngày:** giữ được nhiều hơn, nhưng dễ ghi công cho lead không liên quan.

Câu lệnh dưới đây áp cách 2 với ba mức thời gian: không giới hạn, 90 ngày, 30 ngày. Với mỗi mức, nó đếm:
- số cặp lead–hợp đồng;
- số hợp đồng được ghi công;
- số hợp đồng còn nối với nhiều lead (phải bằng 0 sau xử lý);
- trong các lead được ghi công: bao nhiêu lead có kết quả DEAL (`result = 7`), bao nhiêu lead ghi "không quan tâm" (`result = 1`).

In [ ]:
q('''with l as (select row_number() over () lid, skp_client, code_product_type p, date_assigned, result from leads where code_product_type in ('CL','CC','TW','CD')),
c as (select skp_credit_case, skp_client::BIGINT skp_client, product p, date_apply from contract where product in ('CL','CC','TW','CD') and skp_client is not null),
m as (select l.lid, l.result, c.skp_credit_case, date_diff('day', l.date_assigned, c.date_apply) cach_ngay,
        row_number() over (partition by c.skp_credit_case order by l.date_assigned desc, l.lid) thu_tu
      from l join c on l.skp_client=c.skp_client and l.p=c.p and c.date_apply>=l.date_assigned),
kq as (
  select 'Trước xử lý (mọi cặp)' muc, count(*) so_cap, count(distinct skp_credit_case) so_hd, count(*)-count(distinct skp_credit_case) cap_du, sum((result=7)::int) lead_deal, sum((result=1)::int) lead_khong_quan_tam from m
  union all select 'Sau: 1 lead/hợp đồng, không giới hạn', count(*), count(distinct skp_credit_case), count(*)-count(distinct skp_credit_case), sum((result=7)::int), sum((result=1)::int) from m where thu_tu=1
  union all select 'Sau: 1 lead/hợp đồng, trong 90 ngày', count(*), count(distinct skp_credit_case), count(*)-count(distinct skp_credit_case), sum((result=7)::int), sum((result=1)::int) from m where thu_tu=1 and cach_ngay<=90
  union all select 'Sau: 1 lead/hợp đồng, trong 30 ngày', count(*), count(distinct skp_credit_case), count(*)-count(distinct skp_credit_case), sum((result=7)::int), sum((result=1)::int) from m where thu_tu=1 and cach_ngay<=30)
select * from kq''')

,muc,so_cap,so_hd,cap_du,lead_deal,lead_khong_quan_tam
0,Trước xử lý (mọi cặp),339529,260674,78855,"40,025.00","231,892.00"
1,"Sau: 1 lead/hợp đồng, không giới hạn",260674,260674,0,"38,267.00","174,481.00"
2,"Sau: 1 lead/hợp đồng, trong 90 ngày",219248,219248,0,"36,242.00","138,907.00"
3,"Sau: 1 lead/hợp đồng, trong 30 ngày",179793,179793,0,"35,186.00","103,748.00"


Sau khi giữ một lead cho mỗi hợp đồng, số cặp dư về 0, tức mỗi hợp đồng chỉ còn được ghi công cho đúng một lead.

Bảng cũng cho thấy giới hạn thời gian thay đổi kết quả như thế nào. Từ "không giới hạn" xuống "30 ngày", số lead DEAL được ghi công chỉ giảm từ 38.267 xuống 35.186, tức vẫn giữ khoảng 92%. Trong khi đó số lead "không quan tâm" được ghi công giảm từ 174.481 xuống 103.748, tức bỏ đi khoảng 41%. Nghĩa là giới hạn ngắn chủ yếu loại bỏ những lần chào không liên quan, mà vẫn giữ gần hết những lần chào thật sự dẫn tới hợp đồng. Đây là bằng chứng nghiêng về mức 30 ngày, nhưng vẫn là quyết định của cả nhóm, cần chốt trước khi tính cross-sell. Con số chọn được phải ghi trong slide, kèm phép thử độ nhạy: kết quả xếp hạng shop có đổi nhiều không khi đổi giữa 30 và 90 ngày.

## Phần 10. Tổng hợp: vấn đề, quy mô và cách xử lý đã chọn

Các dòng xếp theo đúng thứ tự vấn đề xuất hiện trong notebook. Cột cuối ghi số hiệu kiểm tra hoặc mục xử lý có bằng chứng.

| Vấn đề | Quy mô | Cách xử lý đã chọn | Bằng chứng |
|---|---|---|---|
| Tên tỉnh cũ | 112 shop (0,8%) | Đổi theo Nghị quyết 202/2025, về đúng 34 tỉnh | 2.2, 9.1 |
| Aldi gần như trùng NS | 3.094/3.095 seller Aldi | Không tách ảnh hưởng của hai yếu tố này | 2.3 |
| Shop trỏ tới seller không tồn tại | 15 shop, mới xuất hiện 17–18/10/2026, chưa có hoạt động | Bỏ khỏi dữ liệu sạch | 3.3 |
| Một tỉnh thuộc nhiều vùng | 7 tỉnh | Không phải lỗi: vùng đi theo tỉnh cũ. Dùng tỉnh mới làm khu vực khi chia peer group | 3.4, 9.2 |
| Shop ghi địa giới cũ (còn tên quận) | 622 shop (4,4%) | Không dùng cột quận; phân tích ở cấp tỉnh mới và vùng | 3.5 |
| Tên phường, xã ghi nhiều cách | 2.315 tên, còn 2.024 sau chuẩn hóa | Chuẩn hóa tên khi ghép dữ liệu cấp phường, xã | 3.6 |
| `SALESROOM_START` không phải ngày mở shop | mọi shop | Gọi là "ngày xuất hiện trong dữ liệu" | 3.7 |
| Dòng trùng emp_2_salesroom | 912 dòng (0,33%) | Bỏ trùng hoàn toàn | 4.1, 9.3 |
| Giá trị quy ước 01/01/3000 | 28% hồ sơ, 85% phân công | `date_sign` đổi thành trống; `dtime_valid_to` đổi thành 17/10/2026 | 4.2, 5.5, 9.4 |
| Cách đếm số SA của shop | Bảng phân công cho trung vị 20 SA mỗi shop, người làm hồ sơ cho trung vị 1 | Đếm SA bằng nhân viên thực sự làm hồ sơ trong tháng | 9.5 |
| CC/BNPL không có `credit_amount` | 79% hợp đồng CC/BNPL đã ký | Dùng `amt_billing` hoặc chỉ đếm số hợp đồng | 5.6 |
| DPD lệch theo thời gian | khoảng 20% hợp đồng đã ký | Chỉ dùng hợp đồng ký 03/2025–07/2026 khi tính trễ kỳ 1 | 5.8, 9.6 |
| Thiếu shop TW trong bảng salesroom | 78% hồ sơ TW | Ghi rõ giới hạn trong slide | 5.9 |
| Doanh số tổng tăng vì thêm shop | 189 → 6.547 shop có hồ sơ | Tính xu hướng và BAU cho từng shop | 5.10 |
| Nhiều dòng mỗi hồ sơ trong contract_product | 4.961 dòng dư (0,63%) | Giữ 1 dòng, ưu tiên dòng đủ thông tin | 6.1, 6.2, 9.7 |
| `goods_group` không thống nhất, trống | 12.476 dòng trống (1,6%) | Gộp tên, điền theo nhóm hàng lớn | 6.3, 9.8 |
| `id_uuid` lặp | 3,4% dòng lead | Không bỏ trùng theo `id_uuid` | 7.1 |
| `code_sa` không phải người | 35,5% dòng lead | Loại khi tính theo nhân viên | 7.2 |
| Hạn mức nhập sai (số hợp đồng nhập vào ô hạn mức, sai đơn vị) | 10 dòng lead | Để trống hạn mức của 10 dòng này; hạn mức 250–281 triệu vẫn giữ | 7.3 |
| Cột kết quả lead | 99,7% lead có `code_lead_status` = 4 | Dùng `result`; hỏi lại Ban tổ chức | 7.4 |
| Lead nối nhiều–nhiều với hợp đồng | 21% hợp đồng nối được có từ 2 lead | Mỗi hợp đồng 1 lead gần nhất, chốt giới hạn ngày | 7.6, 7.7, 9.9 |
| Lead ngoài bảng salesroom | 97% lead sau trọng số | Ghi rõ giới hạn trong slide | 7.8 |